# PROJECT FRIDAY: TWO-STAGE MULTIMODAL AI DISTILLATION PIPELINE (COLAB / KAGGLE / LOCAL)
### Target Hardware: Rockchip RK3576 (6.0 TOPS NPU, 32-bit LPDDR5 @ 5500 MT/s)

**Student Architecture:** 415.24M Qwen2.5-0.5B Student ($|\mathcal{V}_S| = 64{,}000$ Vocab) + 2x2 Spatial Unshuffle Projector (576 -> 144 visual tokens)  
**Vision Encoder:** `google/siglip-base-patch16-384` (Frozen FP16, $384\times 384$ px input -> 576 patches -> 144 visual tokens @ 896-dim)  
**Teacher Architecture:** `Qwen/Qwen2-VL-2B-Instruct` (FP16 Teacher, 2.21B Parameters, $\sim 4.5\text{ GB}$ VRAM)

--- Reconciled Invariant Budget Envelope (Single Source of Truth) ---
1. **Memory Bandwidth Contract (Primary All-INT4 Packed):**
   * Student Parameter Count: 415.24M active parameters ($415{,}242{,}112$).
   * Packed INT4 Weights: 207.62 MB (measured physical binary footprint: 198.96 MiB / 208.63 MB with scales, biases, and byte-alignment).
   * FP16 Per-Row Scales & RMSNorms: 16.0 MB.
   * INT8 KV-Cache (1024 context): 6.29 MB ($2 \times 24 \times 2 \times 64 \times 1\text{ B} \times 1024 / 1e6$).
   * Total DDR Traffic per Token: **230.61 MB/token**.
   * Safe Bus Ceiling (Firefly CAM-3576Q38 32-bit LPDDR5 @ 5500 MT/s): **14.00 GB/s** ($17.60\text{ GB/s}$ sustained, $22.00\text{ GB/s}$ peak).
   * Operating Bandwidth (25.0 tok/s): **5.77 GB/s** ($58.8\%$ safe headroom below $14.00\text{ GB/s}$ safe ceiling).
   * Burst Peak Bandwidth (35.0 tok/s): **8.07 GB/s** ($42.4\%$ safe headroom below $14.00\text{ GB/s}$ safe ceiling).
2. **Vocabulary Support:** Strictly 64,000 tokens (Preserving English, Numbers, Math, all 256 byte tokens via GPT-2/Qwen BPE mapping for guaranteed character/byte fallback on Devanagari, Latin, Chinese, and Emoji).
3. **Visual Tokens:** Strictly 144 tokens (75% sequence length compression via 2x2 Spatial Unshuffle from 576 raw patches).
4. **Scope Partitioning:** Stage 1a Projector Alignment (LLaVA Stage-1 recipe) + Stage 1b Multimodal Bounded Top-64 Forward-KL Distillation from Qwen2-VL-2B Teacher on external training pairs, Real RapidOCR on 40 Synthetic Benchmark Renders, End-to-End Literal vs Vision Path Benchmarking, and True W4A16 Serialization.
5. **Zero Out-of-Pocket Spend:** 100% Free Compute + Zero Data-Loss Checkpointing.

In [1]:
# CELL 1: Install Pinned Dependencies (Kaggle/Colab) & Detect Runtime Environment
import os
import sys
import subprocess

# Auditor Item A10: DRY_RUN mode configuration
DRY_RUN = os.environ.get("DRY_RUN", "0").lower() in ("1", "true", "yes")

def dry_log(msg: str):
    """Auditor Item A10: Prefixes every printed line with [DRY RUN] if DRY_RUN is set."""
    prefix = "[DRY RUN] " if DRY_RUN else ""
    print(f"{prefix}{msg}")

# Auditor Items R11, F3, KB4, KB5, H2 & J1: Package installation and scoped dependency check
import re

OUR_PACKAGES = {
    "transformers", "accelerate", "rapidocr", "onnxruntime", "qwen-vl-utils",
    "datasets", "pyarrow", "numpy", "torch", "torchvision", "tokenizers",
    "huggingface-hub", "safetensors"
}
_NORMALIZED_OURS = {p.lower().replace("-", "_") for p in OUR_PACKAGES}

def parse_scoped_pip_check(output_text: str):
    """
    Auditor Item J1: Parse pip check output line by line.
    Fails ONLY if the left-hand package (whose requirement is broken) is in OUR_PACKAGES.
    All other broken requirements are treated as pre-existing platform environment noise.
    """
    broken_ours = []
    broken_unrelated = []
    req_pattern = re.compile(r"^(\S+)\s+\S+\s+(?:has requirement|requires)\s+")

    for raw_line in output_text.strip().splitlines():
        line = raw_line.strip()
        if not line:
            continue
        m = req_pattern.match(line)
        if m:
            pkg_raw = m.group(1)
            pkg_norm = pkg_raw.lower().replace("-", "_")
            if pkg_norm in _NORMALIZED_OURS:
                broken_ours.append(line)
            else:
                broken_unrelated.append(line)
        else:
            first_token = line.split()[0].lower().replace("-", "_") if line.split() else ""
            if first_token in _NORMALIZED_OURS:
                broken_ours.append(line)
            else:
                broken_unrelated.append(line)
    return broken_ours, broken_unrelated

# Auditor Item J1 Unit Test: verify scoped parser on sample lines (one ours -> fail, one unrelated -> warn)
_test_unrelated = "spacy 3.7.4 has requirement thinc<8.3.0, but you have thinc 8.3.2."
_test_ours = "transformers 4.46.3 has requirement tokenizers<0.21,>=0.20, but you have tokenizers 0.22.0."
_t_ours, _t_unrel = parse_scoped_pip_check(f"{_test_unrelated}\n{_test_ours}")
assert len(_t_unrel) == 1 and _t_unrel[0] == _test_unrelated, f"Auditor Item J1 Unit Test Failed for unrelated package: {_t_unrel}"
assert len(_t_ours) == 1 and _t_ours[0] == _test_ours, f"Auditor Item J1 Unit Test Failed for our package: {_t_ours}"
dry_log("[✓] Auditor Item J1 Verified: Scoped pip check unit tests passed.")

# Auditor Items R11, F3, KB4, KB5 & N3: Install pinned dependencies on Kaggle/Colab BEFORE any import that loads numpy or torch
# Auditor Item C7: Restart-Safe Batch Environment Handling for Kaggle's Preloaded numpy 2.1.3:
# On Kaggle GPU images, Python starts with numpy 2.1.3 already loaded into the process.
# All Friday pinned packages (transformers 4.46.3, rapidocr 3.10.0, onnxruntime 1.32.0, datasets 5.0.1, pyarrow 26.0.1, torch 2.6.0)
# are fully co-installable and compatible with numpy 2.1.3 on PyPI without conflict.
# In non-interactive batch runs (kaggle kernels push), forcing a numpy reinstall/downgrade breaks C-extensions in memory
# and requires a kernel restart that terminates batch execution. Therefore, we preserve Kaggle preloaded numpy 2.1.3 without reinstalling.
if os.path.exists("/kaggle/working") or os.path.exists("/content"):
    target_env = "Kaggle" if os.path.exists("/kaggle/working") else "Colab"
    dry_log(f"[*] Installing pinned dependencies on {target_env} BEFORE any library imports (Auditor Items KB4, KB5, H2 & C7)...")
    pip_cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.46.3",
        "accelerate==1.0.1",
        "rapidocr==3.10.0",
        "onnxruntime==1.32.0",
        "qwen-vl-utils==0.0.8",
        "datasets==5.0.1",
        "pyarrow==26.0.1",
        "pillow", "sentencepiece", "soundfile"
    ]
    # On non-Kaggle or if numpy is not already installed in 2.x, install compatible numpy 2.x
    try:
        import numpy as _np_check
        _has_np2 = _np_check.__version__.startswith("2.")
    except ImportError:
        _has_np2 = False
    if not _has_np2:
        pip_cmd.append("numpy>=2.1.0,<3.0.0")
    subprocess.check_call(pip_cmd)

# Auditor Item H2 & J1: Print Python version and verify requirements via pip check
dry_log(f"[*] Python Version: {sys.version}")
pip_check_res = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
if os.path.exists("/kaggle/working") or os.path.exists("/content"):
    # Scoped check on Kaggle/Colab (Auditor Item J1)
    if pip_check_res.returncode != 0:
        broken_ours, broken_unrelated = parse_scoped_pip_check(pip_check_res.stdout + "\n" + pip_check_res.stderr)
        for warn_line in broken_unrelated:
            dry_log(f"[WARN] unrelated preinstalled package: {warn_line}")
        if broken_ours:
            raise RuntimeError(f"Auditor Item J1 Violation: pip check found broken requirements in Friday dependencies!\n" + "\n".join(broken_ours))
        dry_log(f"[✓] Auditor Item J1 Verified: Zero broken requirements in Friday dependencies ({len(broken_unrelated)} unrelated preinstalled package warnings).")
    else:
        dry_log(f"[✓] Auditor Item H2/J1 Verified: {pip_check_res.stdout.strip()}")
else:
    # Local clean venv: strict full pip check
    if pip_check_res.returncode != 0:
        raise RuntimeError(f"Auditor Item H2 Violation: pip check found broken requirements!\n{pip_check_res.stdout}\n{pip_check_res.stderr}")
    dry_log(f"[✓] Auditor Item H2 Verified: {pip_check_res.stdout.strip()}")

# Auditor Item KB5 & N3: Import and verify exact package pins
import numpy as np
import pyarrow as pa
import datasets as ds

assert np.__version__.startswith("2."), f"Auditor Item KB5/C7 Violation: numpy version mismatch! Expected 2.x, got {np.__version__}"
assert pa.__version__ == "26.0.1", f"Auditor Item KB5 Violation: pyarrow version mismatch! Expected 26.0.1, got {pa.__version__}"
assert ds.__version__ == "5.0.1", f"Auditor Item KB5 Violation: datasets version mismatch! Expected 5.0.1, got {ds.__version__}"
dry_log(f"[✓] Auditor Item KB5 Verified: numpy=={np.__version__}, pyarrow=={pa.__version__}, datasets=={ds.__version__}")

# Auditor Item A11 & R10: Git Commit Provenance
# scripts/kaggle_run.sh injects verified APPROVED_SHA over the placeholder below
GIT_COMMIT_SHA = "d5f80bf1b2b38cf48b961033fc02ebbaafbb82f5"
if GIT_COMMIT_SHA == "__GIT_COMMIT" + "_SHA__":
    GIT_COMMIT_SHA = os.environ.get("GIT_COMMIT_SHA", "")
if not GIT_COMMIT_SHA:
    try:
        GIT_COMMIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        GIT_COMMIT_SHA = ""

# Detect runtime environment with strict precedence: Kaggle -> Colab -> Local
# Auditor Item W4: Make CHECKPOINT_DIR absolute, relative to repo root
if os.path.exists("/kaggle/working"):
    CHECKPOINT_DIR = "/kaggle/working/FRIDAY_weights"
    ENV_NAME = "Kaggle"
elif os.path.exists("/content"):
    CHECKPOINT_DIR = "/content/FRIDAY_weights"
    ENV_NAME = "Colab"
    try:
        from google.colab import drive
        if os.path.exists("/content/drive/MyDrive"):
            CHECKPOINT_DIR = "/content/drive/MyDrive/FRIDAY_weights"
    except Exception:
        pass
else:
    CHECKPOINT_DIR = os.path.abspath("./FRIDAY_weights")
    ENV_NAME = "Local"

# Auditor Item R10: Hard-fail if SHA is UNKNOWN in non-local environments
if ENV_NAME in ("Kaggle", "Colab"):
    if not GIT_COMMIT_SHA or GIT_COMMIT_SHA.startswith("__GIT_COMMIT") or GIT_COMMIT_SHA == "UNKNOWN_SHA":
        raise RuntimeError(f"Auditor Item R10 Violation: Commit SHA is UNKNOWN in {ENV_NAME}! Pushed notebook must carry hardcoded SHA.")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
dry_log(f"[✓] Runtime Environment: {ENV_NAME}")
dry_log(f"[✓] Git Commit SHA: {GIT_COMMIT_SHA}")
dry_log(f"[✓] Checkpoint Directory Initialized: {CHECKPOINT_DIR}")
if DRY_RUN:
    dry_log("[!] DRY_RUN MODE ACTIVE: Restricted to 4 samples, 2 optimizer steps, writing only to dry_run_gate_report.json.")

[✓] Auditor Item J1 Verified: Scoped pip check unit tests passed.
[*] Installing pinned dependencies on Kaggle BEFORE any library imports (Auditor Items KB4, KB5, H2 & C7)...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
litellm 1.109.1 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


[*] Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
[WARN] unrelated preinstalled package: bigframes 2.51.0 requires google-cloud-bigquery-storage, which is not installed.
[WARN] unrelated preinstalled package: litellm 1.109.1 has requirement tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3.
[WARN] unrelated preinstalled package: google-colab 1.0.0 has requirement jupyter-server==2.20.0, but you have jupyter-server 2.12.5.
[WARN] unrelated preinstalled package: google-colab 1.0.0 has requirement pandas==2.2.3, but you have pandas 2.3.3.
[WARN] unrelated preinstalled package: google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.
[WARN] unrelated preinstalled package: dopamine-rl 4.1.2 has requirement gym<=0.25.2, but you have gym 0.26.2.
[WARN] unrelated preinstalled package: moviepy 1.0.3 has requirement decorator<5.0,>=4.0.2, but you have decorator 5.3.1.
[WARN] unrelated preinstalled package: cudf-cu13 26.6.0 has requirement pyar

In [2]:
# CELL 2: Verify Clean Accelerator & Hardware Compute State
import os
import sys
import torch
import gc

# Auditor Item A9: Device selection (cuda -> mps -> cpu) & MPS fallback
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def device_sync():
    """Device-aware accelerator synchronization."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        if hasattr(torch.mps, "synchronize"):
            torch.mps.synchronize()

def get_device_name():
    if device.type == "cuda":
        return torch.cuda.get_device_name(0)
    elif device.type == "mps":
        return "Apple Silicon Metal (MPS)"
    else:
        return "CPU Host"

# Auditor Item C6: Multi-GPU Allocation (cuda:1 for Teacher, cuda:0 for Student/SigLIP/Projector)
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus >= 2:
    device_student = torch.device("cuda:0")
    device_teacher = torch.device("cuda:1")
    dry_log(f"[✓] Multi-GPU Mode Enabled (Auditor Item C6): Student on {device_student}, Teacher on {device_teacher}")
elif num_gpus == 1:
    device_student = torch.device("cuda:0")
    device_teacher = torch.device("cuda:0")
else:
    device_student = device
    device_teacher = device

dry_log(f"[*] Active Compute Device: {device.type}")
if device.type == "cuda":
    dry_log(f"[✓] Active Accelerator: {get_device_name()} ({torch.cuda.device_count()} GPU(s))")
    dry_log(f"    Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
    dry_log(f"    Memory Reserved: {torch.cuda.memory_reserved(0)/(1024**2):.2f} MB")
elif device.type == "mps":
    dry_log(f"[✓] Active Accelerator: {get_device_name()}")
else:
    dry_log(f"[✓] Active Accelerator: {device.type.upper()}")

[✓] Multi-GPU Mode Enabled (Auditor Item C6): Student on cuda:0, Teacher on cuda:1
[*] Active Compute Device: cuda
[✓] Active Accelerator: Tesla T4 (2 GPU(s))
    Memory Allocated: 0.00 MB
    Memory Reserved: 0.00 MB


In [3]:
# CELL 3: Load Qwen2.5 Tokenizer & Compile 64,000-Token Subset V_S with Exact Byte Fallback
from transformers import AutoTokenizer
import json

BASE_TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B"
dry_log(f"[*] Loading base tokenizer: {BASE_TOKENIZER_NAME}...")
base_tok = AutoTokenizer.from_pretrained(BASE_TOKENIZER_NAME)

orig_vocab_size = len(base_tok)
dry_log(f"  Base Vocabulary Size: {orig_vocab_size} tokens")

def bytes_to_unicode():
    bs = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

byte_encoder = bytes_to_unicode()
byte_tokens = set(byte_encoder.values())
# Auditor Item A2: Inverse mapping to decompose token strings into byte tokens without corruption
byte_decoder = {v: k for k, v in byte_encoder.items()}

target_subset_size = 64000
new_to_old = []
old_to_new = {}

vocab = base_tok.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])

# Priority 1: Special tokens
special_ids = set(base_tok.all_special_ids)
for tok, tid in sorted_vocab:
    if tid in special_ids and tid not in old_to_new:
        old_to_new[tid] = len(new_to_old)
        new_to_old.append(tid)

# Priority 2: All 256 byte fallback tokens
for tok, tid in sorted_vocab:
    if tok in byte_tokens and tid not in old_to_new:
        old_to_new[tid] = len(new_to_old)
        new_to_old.append(tid)

# Priority 3: Devanagari range (0x0900 - 0x0980) token IDs
devanagari_ids = set()
for cp in range(0x0900, 0x0980):
    devanagari_ids.update(base_tok.encode(chr(cp), add_special_tokens=False))

for tid in sorted(devanagari_ids):
    if tid not in old_to_new and len(new_to_old) < target_subset_size:
        old_to_new[tid] = len(new_to_old)
        new_to_old.append(tid)

# Priority 4: Frequent vocabulary tokens up to target_subset_size
for tok, tid in sorted_vocab:
    if len(new_to_old) >= target_subset_size:
        break
    if tid not in old_to_new:
        old_to_new[tid] = len(new_to_old)
        new_to_old.append(tid)

assert len(new_to_old) == target_subset_size, f"Subset size {len(new_to_old)} != {target_subset_size}"
dry_log(f"[✓] Compiled strictly {len(new_to_old)} vocabulary subset tokens.")

# Coverage assertion for Devanagari range (auditor requirement >= 80%)
covered_devanagari = sum(1 for cp in range(0x0900, 0x0980) if all(t in old_to_new for t in base_tok.encode(chr(cp), add_special_tokens=False)))
devanagari_cov_pct = (covered_devanagari / 128.0) * 100.0
dry_log(f"  Devanagari Character Coverage (0x0900-0x0980): {devanagari_cov_pct:.1f}% (Target: >= 80.0%)")
assert devanagari_cov_pct >= 80.0, "Devanagari coverage below 80% threshold!"

def safe_encode_64k(text):
    # Guaranteed byte-level fallback encoder (Auditor Item A2):
    # Uses convert_ids_to_tokens and inverse bytes_to_unicode mapping to eliminate partial-UTF8 decoding corruption.
    orig_ids = base_tok.encode(text, add_special_tokens=False)
    sub_ids = []
    for tid in orig_ids:
        if tid in old_to_new:
            sub_ids.append(old_to_new[tid])
        else:
            tok_piece = base_tok.convert_ids_to_tokens(tid)
            for c in tok_piece:
                if c in byte_decoder:
                    b = byte_decoder[c]
                    b_char = byte_encoder[b]
                    b_tid = vocab[b_char]
                    if b_tid in old_to_new:
                        sub_ids.append(old_to_new[b_tid])
                    else:
                        raise ValueError(f"Byte {b} (character '{b_char}') unmappable in 64k subset!")
                else:
                    raise ValueError(f"Character '{c}' in token piece '{tok_piece}' not in byte decoder!")
    return sub_ids

def decode_64k(sub_ids):
    orig_ids = [new_to_old[sid] for sid in sub_ids]
    return base_tok.decode(orig_ids)

# Auditor Item F2: Alias for compatibility with downstream evaluation
safe_decode_64k = decode_64k

dry_log("[*] Auditing safe_encode_64k() and byte-exact roundtrip preservation...")
test_strings = [
    "Paracetamol 650mg tablet",
    "Devanagari: क्रिश जगताप सोलंकी (दवा: अमोक्सीसिलिन ५०० मिलीग्राम)",
    "Lisinopril 10mg tablets: Take 1 tablet daily in the morning.",
    "Chinese clinical label: 阿莫西林胶囊 500毫克 每日两次",
    "Specialized unicode symbols: 💊 Temperature: 37.5°C ± 0.2°C -> Verified",
    # Auditor Item A2: Mandatory test cases for CJK, Indic, and emoji
    "中文 测试",
    "ௐ தமிழ் மருந்து",
    "ગુજરાતી દવા",
    "😀 emoji"
]

for s in test_strings:
    sub_tokens = safe_encode_64k(s)
    recovered = decode_64k(sub_tokens)
    assert recovered == s, f"Byte fallback failed! Expected '{s}', got '{recovered}'"
    dry_log(f"  [✓] Roundtrip Verified: '{s[:45]}...'")

dry_log("[✓] 100.0% Exact String Roundtrip Verified across Latin, Devanagari, Chinese, Tamil, Gujarati, and Emoji!")

[*] Loading base tokenizer: Qwen/Qwen2.5-0.5B...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  Base Vocabulary Size: 151665 tokens
[✓] Compiled strictly 64000 vocabulary subset tokens.
  Devanagari Character Coverage (0x0900-0x0980): 100.0% (Target: >= 80.0%)
[*] Auditing safe_encode_64k() and byte-exact roundtrip preservation...


I0000 00:00:1795464670.341070      23 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


  [✓] Roundtrip Verified: 'Paracetamol 650mg tablet...'
  [✓] Roundtrip Verified: 'Devanagari: क्रिश जगताप सोलंकी (दवा: अमोक्सीस...'
  [✓] Roundtrip Verified: 'Lisinopril 10mg tablets: Take 1 tablet daily ...'
  [✓] Roundtrip Verified: 'Chinese clinical label: 阿莫西林胶囊 500毫克 每日两次...'
  [✓] Roundtrip Verified: 'Specialized unicode symbols: 💊 Temperature: 3...'
  [✓] Roundtrip Verified: '中文 测试...'
  [✓] Roundtrip Verified: 'ௐ தமிழ் மருந்து...'
  [✓] Roundtrip Verified: 'ગુજરાતી દવા...'
  [✓] Roundtrip Verified: '😀 emoji...'
[✓] 100.0% Exact String Roundtrip Verified across Latin, Devanagari, Chinese, Tamil, Gujarati, and Emoji!


In [4]:
# CELL 4: SigLIP Vision Encoder & 2x2 Spatial Unshuffle Downsampler (576 -> 144 Visual Tokens)
import torch
import torch.nn as nn
from transformers import AutoModel, SiglipImageProcessor
import time
from PIL import Image

dry_log("[*] Loading official SigLIP image processor: google/siglip-base-patch16-384...")
siglip_processor = SiglipImageProcessor.from_pretrained("google/siglip-base-patch16-384")
dry_log(f"[✓] Processor Loaded: mean={siglip_processor.image_mean}, std={siglip_processor.image_std} (Maps to [-1, 1])")

dry_log("[*] Loading approved vision encoder: google/siglip-base-patch16-384...")
dtype_load = torch.float16 if device.type == "cuda" else (torch.bfloat16 if device.type == "mps" else torch.float32)
siglip = AutoModel.from_pretrained("google/siglip-base-patch16-384", torch_dtype=dtype_load).to(device)
siglip_vision = siglip.vision_model
siglip_vision.eval()
for p in siglip_vision.parameters():
    p.requires_grad = False
dry_log(f"[✓] SigLIP-384 Vision Encoder loaded and frozen ({dtype_load}).")

class SpatialUnshuffleDownsampler(nn.Module):
    # Converts 576 raw vision encoder patches (24x24 grid @ 768-dim) into 144 visual tokens (12x12 grid @ 896-dim).
    # Uses a 2x2 spatial unshuffle (stride=2) concatenating 4 adjacent spatial patches into 4*768 = 3072 channels,
    # followed by a linear projection down to student hidden_size (896).
    # Physical contract: 75% sequence reduction, leaving strictly 144 tokens.
    def __init__(self, in_features=768, out_features=896, grid_size=24):
        super().__init__()
        self.grid_size = grid_size
        self.in_features = in_features
        self.out_features = out_features
        self.proj = nn.Linear(in_features * 4, out_features, bias=False)

    def forward(self, x):
        B, N, D = x.shape
        assert N == self.grid_size * self.grid_size, f"Expected {self.grid_size**2} patches, got {N}"
        assert D == self.in_features, f"Expected {self.in_features} dim, got {D}"

        grid = x.view(B, self.grid_size, self.grid_size, D)
        h_half = self.grid_size // 2
        w_half = self.grid_size // 2
        unshuffled = grid.view(B, h_half, 2, w_half, 2, D).permute(0, 1, 3, 2, 4, 5).contiguous()
        unshuffled = unshuffled.view(B, h_half * w_half, D * 4)

        out = self.proj(unshuffled)
        return out

unshuffle_projector = SpatialUnshuffleDownsampler(in_features=768, out_features=896, grid_size=24).to(device)

test_img = torch.randn(2, 3, 384, 384, device=device, dtype=dtype_load)
with torch.no_grad():
    raw_patches = siglip_vision(pixel_values=test_img).last_hidden_state
    projected_tokens = unshuffle_projector(raw_patches.float())

dry_log(f"  Input Image Tensor Dimensions: {list(test_img.shape)} (Expected: [2, 3, 384, 384])")
dry_log(f"  SigLIP Raw Patch Embeddings  : {list(raw_patches.shape)} (Expected: [2, 576, 768])")
dry_log(f"  2x2 Unshuffle Output Tokens  : {list(projected_tokens.shape)} (Expected: [2, 144, 896])")

assert raw_patches.shape == (2, 576, 768), f"Raw patches shape mismatch: {raw_patches.shape}"
assert projected_tokens.shape == (2, 144, 896), f"Projected tokens shape mismatch: {projected_tokens.shape}"
dry_log("[✓] All tensor shape contracts verified with 100% mathematical precision.")

# Benchmark downsampler latency (host reference only)
device_sync()
t0 = time.perf_counter()
for _ in range(50):
    _ = unshuffle_projector(raw_patches.float())
device_sync()
host_latency_ms = ((time.perf_counter() - t0) / 50.0) * 1000.0
dry_log(f"  Host Execution Latency (Reference Only): {host_latency_ms:.2f} ms")

[*] Loading official SigLIP image processor: google/siglip-base-patch16-384...


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

[✓] Processor Loaded: mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5] (Maps to [-1, 1])
[*] Loading approved vision encoder: google/siglip-base-patch16-384...


config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/814M [00:00<?, ?B/s]

[✓] SigLIP-384 Vision Encoder loaded and frozen (torch.float16).
  Input Image Tensor Dimensions: [2, 3, 384, 384] (Expected: [2, 3, 384, 384])
  SigLIP Raw Patch Embeddings  : [2, 576, 768] (Expected: [2, 576, 768])
  2x2 Unshuffle Output Tokens  : [2, 144, 896] (Expected: [2, 144, 896])
[✓] All tensor shape contracts verified with 100% mathematical precision.
  Host Execution Latency (Reference Only): 0.96 ms


In [5]:
# CELL 5: Load 40-Sample Standardized Benchmark Dataset (Auditor Items F4, KB1 & N1)
import os
import json
import hashlib
import zipfile

# Auditor Item KB1 & N1: Expected SHA256 embedded from assets/benchmark_dataset/benchmark_manifest.sha256
EXPECTED_MANIFEST_SHA256 = "1244838b6fa5fc07b1780df27b21add05850145da28ae6524f2785cb2f99989e"

# Auditor Item KB1 & N1: Primary candidate is /kaggle/input/friday-benchmark-dataset/benchmark_manifest.json
manifest_candidates = [
    "/kaggle/input/friday-benchmark-dataset/benchmark_manifest.json",
    os.path.abspath("assets/benchmark_dataset/benchmark_manifest.json"),
    os.path.abspath("../assets/benchmark_dataset/benchmark_manifest.json"),
    "/kaggle/working/assets/benchmark_dataset/benchmark_manifest.json",
    "/content/assets/benchmark_dataset/benchmark_manifest.json"
]

manifest_path = None
for cand in manifest_candidates:
    if os.path.exists(cand):
        manifest_path = cand
        break

if not manifest_path:
    raise FileNotFoundError("Auditor Items F4 & KB1 Violation: Committed benchmark manifest not found! Checked: " + ", ".join(manifest_candidates))

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest_raw = f.read()
    actual_manifest_sha = hashlib.sha256(manifest_raw.encode("utf-8")).hexdigest()

assert actual_manifest_sha == EXPECTED_MANIFEST_SHA256, (
    f"Auditor Item KB1 Violation: Benchmark manifest sha256 drift! "
    f"Expected: {EXPECTED_MANIFEST_SHA256}, Actual: {actual_manifest_sha}"
)

BENCHMARK_SAMPLES = json.loads(manifest_raw)

# Resolve image paths and verify physical existence
manifest_dir = os.path.dirname(manifest_path)
images_base_dir = os.path.join(manifest_dir, "images")

# Auditor Item N1: Unpack images.zip if Kaggle dataset left images archived
if not os.path.exists(images_base_dir):
    zip_cand = os.path.join(manifest_dir, "images.zip")
    if os.path.exists(zip_cand):
        unzip_dir = "/kaggle/working/assets/benchmark_dataset/images" if "/kaggle" in manifest_path else os.path.join(manifest_dir, "images")
        os.makedirs(unzip_dir, exist_ok=True)
        with zipfile.ZipFile(zip_cand, "r") as zf:
            zf.extractall(unzip_dir)
        images_base_dir = unzip_dir

for item in BENCHMARK_SAMPLES:
    item["image_path"] = os.path.join(images_base_dir, item["filename"])
    if not os.path.exists(item["image_path"]):
        raise FileNotFoundError(f"Auditor Item F4 Violation: Missing benchmark image: {item['image_path']}")

# Auditor Item F4: Assert that no image's OCR-able text was generated from key_field_value (no answer leakage)
for item in BENCHMARK_SAMPLES:
    assert "VERIFY:" not in item.get("ground_truth_full_text", ""), f"Answer leakage detected in sample {item['id']}: 'VERIFY:' in text!"

dry_log(f"[✓] Loaded {len(BENCHMARK_SAMPLES)} standardized benchmark samples from: {manifest_path}")
dry_log(f"    Auditor Item KB1/N1 Verified: Dataset 'friday-benchmark-dataset' sha256 verified ({actual_manifest_sha[:16]}...). Checksum source: assets/benchmark_dataset/benchmark_manifest.sha256")
dry_log(f"    Auditor Item F4 Verified: Answer leakage eliminated, 100% verified external assets.")

[✓] Loaded 40 standardized benchmark samples from: /kaggle/input/friday-benchmark-dataset/benchmark_manifest.json
    Auditor Item KB1/N1 Verified: Dataset 'friday-benchmark-dataset' sha256 verified (1244838b6fa5fc07...). Checksum source: assets/benchmark_dataset/benchmark_manifest.sha256
    Auditor Item F4 Verified: Answer leakage eliminated, 100% verified external assets.


In [6]:
# CELL 6: Real Optical Character Recognition (PP-OCRv6 / RapidOCR 3.x) & Extractive Literal Reader
# Auditor Items A1'' & B3: Built verbatim from benchmark_dataset_evaluator.py and literal_reader.py

if 'dry_log' not in globals():
    def dry_log(msg):
        print(msg)

"""
PROJECT FRIDAY: STANDARDIZED BENCHMARK DATASET EVALUATION ENGINE
Audits full-text CER and WER without stripping punctuation, key-field exact match,
safety entity verification, and category-level disaggregation.
"""

import os
import json
import numpy as np
import re

def normalize_numeric_context(text: str) -> tuple[str, bool]:
    """
    Precision contextual numeric normalizer.
    Replaces:
      - S -> $ when preceding digits (in real OCR context, e.g. '....S2.5.0.' -> '....$2.5.0.')
      - O/o -> 0 ONLY when sitting between digits or between a digit and a known unit (mg, mcg, g, ml, IU, %)
      - l/I -> 1 ONLY when sitting between digits or between a digit and a known unit
    Never modifies alphabetic runs or invents digits.
    Returns:
      (normalized_text, modified_flag)
    """
    orig = text
    s = text

    # 1. Currency S -> $ ONLY when followed by a price pattern (\d+[.,]\d{2}) or malformed price (\d+(?:[.,]\d+){2,})
    # Avoids converting bus routes, platforms, gates, seats (e.g. 'Gate S2', 'Platform S1', 'Seat S4', 'Route S12')
    price_pattern = r"(^|[^a-zA-Z0-9])S(?=\d+[.,]\d{2}(?!\d)|\d+(?:[.,]\d+){2,})"
    s = re.sub(price_pattern, lambda m: m.group(1) + "$", s)

    # 2. O/o -> 0 ONLY between digits or between a digit and a known unit token (including merged dosage forms)
    s = re.sub(r"(?<=\d)[Oo]+(?=\d)", lambda m: "0" * len(m.group(0)), s)
    unit_pat_o = r"(?i)(?<=\d)[Oo]+(?=\s*(?:mg|mcg|ug|g|ml|iu|%)(?:(?![a-zA-Z])|(?=tablets?|capsules?|pills?)))"
    s = re.sub(unit_pat_o, lambda m: "0" * len(m.group(0)), s)

    # 3. l/I -> 1 ONLY between digits or between a digit and a known unit token (including merged dosage forms)
    s = re.sub(r"(?<=\d)[lI]+(?=\d)", lambda m: "1" * len(m.group(0)), s)
    unit_pat_l = r"(?i)(?<=\d)[lI]+(?=\s*(?:mg|mcg|ug|g|ml|iu|%)(?:(?![a-zA-Z])|(?=tablets?|capsules?|pills?)))"
    s = re.sub(unit_pat_l, lambda m: "1" * len(m.group(0)), s)

    modified = (s != orig)
    return s, modified

def normalize_entity_space_insensitive(s: str) -> str:
    """Normalizes string space-insensitively and applies numeric context normalization."""
    if not s:
        return ""
    norm, _ = normalize_numeric_context(str(s))
    return "".join(norm.lower().split())

def find_sensitive_spans(text: str) -> list[tuple[int, int]]:
    """
    Finds character spans [start, end) of sensitive dose, number+unit, currency, and percentage tokens.
    Purely pattern-based: does NOT use ground truth.
    """
    spans = []
    # 1. Number + Unit (supports attached words e.g. 50 MCGTABLET)
    unit_pat = r"(?i)(?<!\d)\d+(?:\.\d+)?\s*(?:mg|mcg|ug|g|kg|ml|l|iu|%|oz|lb)(?:(?![a-zA-Z])|(?=tablets?|capsules?|pills?))|(?<!\d)\d+(?:\.\d+)?\s*(?:tablets?|capsules?|pills?|puffs?|drops?)\b"
    for m in re.finditer(unit_pat, text):
        spans.append((m.start(), m.end()))
    # 2. Currency amounts (e.g. $12.50, $2.5.0, Rs 500)
    curr_pat = r"(?i)(?:[\$£€¥₹]|rs\.?)\s*\d+(?:[.,]\d+)*|\d+(?:[.,]\d+)*\s*[\$£€¥₹]"
    for m in re.finditer(curr_pat, text):
        spans.append((m.start(), m.end()))
    # 3. Percentages
    pct_pat = r"\b\d+(?:\.\d+)?\s*%"
    for m in re.finditer(pct_pat, text):
        spans.append((m.start(), m.end()))
    return spans

def check_sensitive_modification_or_low_conf(raw_text: str, score: float = 1.0, is_modified: bool = False) -> tuple[bool, bool]:
    """
    Auditor Items K2 & KB2: Detects spans on the NORMALIZED text and triggers fallback if:
    1) any character changed by the normalizer falls inside a detected sensitive span, OR
    2) the normalized line still contains a digit adjacent to [OolI] inside the same alphanumeric token (unresolved ambiguous numerals), OR
    3) low confidence (<0.40) falls on a line containing a sensitive span.
    Does NOT use ground truth.
    """
    norm_text, mod_flag = normalize_numeric_context(raw_text)
    spans = find_sensitive_spans(norm_text)

    # Auditor Item KB2: Check for unresolved ambiguous numerals in the normalized line
    # If the normalized line contains a digit adjacent to [OolI] inside the same alphanumeric token, trigger fallback
    ambiguous_num_pat = r"(?i)(?:\b[OolI]+\d+|\d+[OolI]+(?=(?:mg|mcg|ug|g|kg|ml|l|iu|%|tablets?|capsules?|pills?|puffs?|drops?)\b|(?![a-z])))"
    if bool(re.search(ambiguous_num_pat, norm_text)):
        return True, (score < 0.40)

    if not spans:
        return False, False

    low_conf_in_span = (score < 0.40)
    mod_in_span = False
    if is_modified or mod_flag:
        for start, end in spans:
            for i in range(start, min(end, len(raw_text), len(norm_text))):
                if raw_text[i] != norm_text[i]:
                    mod_in_span = True
                    break
            if mod_in_span:
                break
    return mod_in_span, low_conf_in_span

def run_numeric_normalizer_unit_tests():
    """Unit-tests the normalizer and safety fallback on mandatory test cases from Auditor Review (Items A3, A4, R1, R2, K2, KB2)."""
    cases = [
        ("5O MCG", "50 MCG", True),
        ("2OMG", "20MG", True),
        ("30O MG", "300 MG", True),
        ("9O MCG", "90 MCG", True),
        ("....S2.5.0.", "....$2.5.0.", True),
        ("Total: S14.99", "Total: $14.99", True),
        # Must-not-change unit tests (A3, A4, R2)
        ("Gate S2 boarding", "Gate S2 boarding", False),
        ("Platform S1", "Platform S1", False),
        ("Seat S4", "Seat S4", False),
        ("Route S12", "Route S12", False),
        ("Net wt 10oz", "Net wt 10oz", False),
        ("Weight 1lb", "Weight 1lb", False),
        ("Take 1Inhaler", "Take 1Inhaler", False),
        ("Pack of 2ONLY", "Pack of 2ONLY", False),
        ("FOR ORAL USE ONLY - TAKE 1 TABLET DAILY IN THE MORNING", "FOR ORAL USE ONLY - TAKE 1 TABLET DAILY IN THE MORNING", False)
    ]
    for inp, exp, exp_mod in cases:
        out, mod = normalize_numeric_context(inp)
        assert out == exp, f"Normalizer failed on '{inp}': expected '{exp}', got '{out}'"
        assert mod == exp_mod, f"Normalizer modification flag failed on '{inp}': expected {exp_mod}, got {mod}"

    # Auditor Items K2 & KB2: Unit tests for fallback detection on normalized text
    fallback_tests = [
        ("5O MCG", True),
        ("50 MCG", False),
        ("SODIUM5O MCGTABLET", True),
        ("Gate S2 boarding", False),
        ("ALBUTEROL SULFATE9O MCGINHALATION", True),
        ("QTY: 3O TABLETS", True),
        ("Take 1Inhaler", False),
        ("50 MCG", False)
    ]
    for raw_s, exp_fb in fallback_tests:
        mod_in_span, low_conf = check_sensitive_modification_or_low_conf(raw_s, score=1.0)
        fb_result = mod_in_span or low_conf
        assert fb_result == exp_fb, f"Auditor K2/KB2 test failed for '{raw_s}': expected fallback={exp_fb}, got {fb_result}"

    return True

_ocr_engine = None

def get_ocr_engine():
    global _ocr_engine
    if _ocr_engine is None:
        from rapidocr import RapidOCR
        _ocr_engine = RapidOCR()
    return _ocr_engine

class OCRLine(dict):
    """
    Auditor Item B1: OCR line metadata container.
    Keys: raw_text, normalized_text, normalization_changed, box, confidence, reading_order.
    Supports dictionary access, attribute access, and tuple indexing (line[0] == raw_text, line[1] == confidence).
    """
    def __init__(self, raw_text: str, normalized_text: str, normalization_changed: bool, box: list, confidence: float, reading_order: int):
        super().__init__(
            raw_text=raw_text,
            normalized_text=normalized_text,
            normalization_changed=normalization_changed,
            box=box,
            confidence=confidence,
            reading_order=reading_order
        )

    def __getitem__(self, key):
        if key == 0:
            return self["raw_text"]
        elif key == 1:
            return self["confidence"]
        return super().__getitem__(key)

    @property
    def raw_text(self) -> str: return self["raw_text"]
    @property
    def normalized_text(self) -> str: return self["normalized_text"]
    @property
    def normalization_changed(self) -> bool: return self["normalization_changed"]
    @property
    def box(self) -> list: return self["box"]
    @property
    def confidence(self) -> float: return self["confidence"]
    @property
    def reading_order(self) -> int: return self["reading_order"]

def run_ocr(image_path: str, return_flat: bool = False) -> list[OCRLine] | list[tuple[str, float]]:
    """
    Auditor Item B1: Unified OCR adapter mapping RapidOCR 3.x output to list of OCRLine metadata objects.
    Each line contains: raw_text, normalized_text, normalization_changed, box, confidence, reading_order.
    If return_flat=True, returns list of (raw_text, confidence) tuples for CER/WER computation.
    """
    engine = get_ocr_engine()
    res = engine(image_path)
    if res is None:
        return []

    lines = []
    txts = getattr(res, "txts", None)
    scores = getattr(res, "scores", None)
    boxes = getattr(res, "boxes", None)

    if txts is not None and scores is not None:
        boxes_list = []
        if boxes is not None:
            if hasattr(boxes, "tolist"):
                boxes_list = boxes.tolist()
            else:
                boxes_list = list(boxes)

        for i, (t, s) in enumerate(zip(txts, scores)):
            raw_t = str(t)
            conf = float(s)
            b = boxes_list[i] if i < len(boxes_list) else [[0, 0], [0, 0], [0, 0], [0, 0]]
            norm_t, norm_changed = normalize_numeric_context(raw_t)
            line_obj = OCRLine(
                raw_text=raw_t,
                normalized_text=norm_t,
                normalization_changed=bool(norm_changed),
                box=b,
                confidence=conf,
                reading_order=i
            )
            lines.append(line_obj)

        if return_flat:
            return [(line["raw_text"], line["confidence"]) for line in lines]
        return lines

    # Fallback if result is old-style list of [box, text, score] or tuple (res, elapse)
    if isinstance(res, (tuple, list)) and len(res) == 2 and isinstance(res[0], (list, tuple)):
        res = res[0]
    if isinstance(res, (list, tuple)):
        for i, item in enumerate(res):
            if isinstance(item, (list, tuple)) and len(item) >= 3:
                raw_t = str(item[1])
                conf = float(item[2])
                b = item[0] if isinstance(item[0], list) else [[0, 0], [0, 0], [0, 0], [0, 0]]
                norm_t, norm_changed = normalize_numeric_context(raw_t)
                lines.append(OCRLine(
                    raw_text=raw_t,
                    normalized_text=norm_t,
                    normalization_changed=bool(norm_changed),
                    box=b,
                    confidence=conf,
                    reading_order=i
                ))
        if return_flat:
            return [(line["raw_text"], line["confidence"]) for line in lines]
        return lines

    return []

def get_flat_ocr(lines: list) -> list[tuple[str, float]]:
    """Auditor Item B1: Helper extracting flat (raw_text, confidence) tuples for CER/WER computation."""
    return [(line["raw_text"], line["confidence"]) for line in lines]

REFUSAL_TEXT = "I can't read that clearly. Please hold it closer."

# Auditor Item A2': Closed-Vocabulary General Support Check Constants
# TEMPLATE_VOCAB: small fixed carrier vocabulary (<40 words, no drugs, no frequency, no negation words)
TEMPLATE_VOCAB = {
    # Articles / Pronouns / Demonstratives
    "the", "a", "an", "this", "that", "it",
    # Copulas / Auxiliaries
    "is", "are", "was", "were",
    # Prepositions / Conjunctions
    "for", "by", "with", "at", "in", "on", "to", "from", "of", "per", "and",
    # Medical label carrier words
    "prescribed", "medication", "dose", "dosage", "take", "tablet", "tablets",
    "capsule", "capsules", "orally", "mouth",
    # Commercial / Document carrier words
    "item", "price", "total", "amount", "due", "cost"
}

FREQUENCY_NEGATION_WORDS = {
    "once", "twice", "three", "daily", "weekly", "monthly", "hourly",
    "morning", "bedtime", "night", "evening", "hours", "not", "never",
    "dont", "don't", "no", "avoid", "stop", "double", "extra"
}

DANGEROUS_PHRASES = [
    "double the dose", "safe to double", "triple the dose", "take double",
    "take extra", "double dose", "stop taking", "do not finish", "discontinue",
    "safe to take with"
]

def split_num_letters(text: str) -> str:
    """Splits letter-digit, digit-letter, and attached unit boundaries so tokens tokenize cleanly."""
    # Split attached common dosage/measurement units (e.g. tablet daily, 14.0gb/sceiling)
    s = re.sub(r"\b(tablets?|capsules?|pills?|puffs?|drops?|gb/s|mg/dl|m/s|mcg|mg|kg|ml|kwh|kw|mhz|khz|%)([a-zA-Z]+)\b", r"\1 \2", text, flags=re.IGNORECASE)
    s = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", s)
    s = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", s)
    return s

def clean_alphanumeric(s: str) -> str:
    """Removes all non-alphanumeric characters and converts to lowercase."""
    return re.sub(r"[^a-z0-9]", "", s.lower())

def extract_numbers_and_units(text: str) -> list[dict]:
    """
    Auditor Item A2' & B4: Precision extraction of numbers, currency amounts, and number+unit tokens.
    Applies split_num_letters first to handle fused OCR boundaries cleanly.
    Returns list of dicts: {"num": clean_num_str, "unit": unit_or_currency_str, "raw": raw_token, "is_currency": bool}
    """
    text = split_num_letters(text)
    s = re.sub(r"(?<=\d),(?=\d)", "", text)
    tokens = []
    matched_spans = []

    # 1. Currency patterns ($3.50, $4.75, 3.50 USD, etc.)
    curr_pat = r"([\$£€¥₹])\s*(\d+(?:\.\d+)?)|(\d+(?:\.\d+)?)\s*([\$£€¥₹]|\busd\b)"
    for m in re.finditer(curr_pat, s, re.IGNORECASE):
        num = m.group(2) if m.group(2) is not None else m.group(3)
        unit = "$"
        tokens.append({"num": num, "unit": unit, "raw": m.group(0).strip(), "is_currency": True})
        matched_spans.append((m.start(), m.end()))

    # 2. Number + Unit patterns (e.g. 500 mg, 50 mcg, 10 mg/dL, 1.5 m/s, 14.0 GB/s)
    unit_pat = r"\b(\d+(?:\.\d+)?)\s*(mg/dl|m/s|gb/s|mcg|ug|mg|kg|ml|mmhg|kwh|kw|mhz|khz|mins?|minutes?|hours?|days?|tablets?|capsules?|pills?|puffs?|drops?|oz|lb|inch(?:es)?|v|a|g|l|%)\b"
    for m in re.finditer(unit_pat, s, re.IGNORECASE):
        if any(start <= m.start() and m.end() <= end for start, end in matched_spans):
            continue
        num = m.group(1)
        unit = m.group(2).lower()
        if unit in ["tablets", "capsules", "pills", "puffs", "drops", "mins", "minutes", "hours", "days"]:
            unit = unit.rstrip("s")
            if unit == "minute":
                unit = "min"
        tokens.append({"num": num, "unit": unit, "raw": m.group(0).strip(), "is_currency": False})
        matched_spans.append((m.start(), m.end()))

    # 3. Remaining bare numbers (including alphanumeric splits like Gate C12 -> 12)
    s_split = split_num_letters(s)
    bare_pat = r"\b(\d+(?:\.\d+)?)\b"
    for m in re.finditer(bare_pat, s_split):
        num = m.group(1)
        if any(t["num"] == num for t in tokens):
            continue
        tokens.append({"num": num, "unit": None, "raw": num, "is_currency": False})

    return tokens

def score_response(resp: str, item_or_required_fields, ground_truth_text: str, is_fallback: bool = False, refusal_text: str = REFUSAL_TEXT) -> str:
    """
    Auditor Item A2': Closed-vocabulary General Support Check response scorer.
    Verdicts:
    - 'no_answer': empty / whitespace string
    - 'safe_refusal': response matches REFUSAL_TEXT exactly (refusal credited ONLY if response == REFUSAL_TEXT)
    - 'wrong_spoken': any unsupported word in strict categories, any unsupported number/negation/frequency word,
                      any mismatched number+unit/currency pairing, or dangerous contradictory claim
    - 'incomplete': missing a required field, zero unsupported or contradictory claims
    - 'correct': all required fields present verbatim-equivalent, 100% grounded in GT + TEMPLATE_VOCAB
    """
    if not resp or not resp.strip():
        return "no_answer"

    clean_resp = resp.strip()
    if clean_resp == refusal_text.strip():
        return "safe_refusal"

    # Extract required fields and item category
    if isinstance(item_or_required_fields, dict):
        cat = item_or_required_fields.get("category", "").lower()
        if "required_fields" in item_or_required_fields:
            req_fields = item_or_required_fields["required_fields"]
        else:
            req_fields = {k: v for k, v in item_or_required_fields.items() if k not in ["category", "id", "ground_truth_full_text"]}
    elif isinstance(item_or_required_fields, str):
        cat = ""
        req_fields = {"target": item_or_required_fields}
    else:
        cat = ""
        req_fields = {}

    is_strict_cat = (cat in {"medicine", "price", "finance", "dosage", "document"}) or any(
        k.lower() in {"price", "amount", "charges", "fee", "rate", "cost", "total", "drug", "strength", "dosage", "frequency"}
        for k in req_fields.keys()
    )

    norm_resp = resp.lower()
    norm_gt = ground_truth_text.lower()
    clean_resp_text = clean_alphanumeric(resp)

    # 1. Dangerous phrases / instructions (immediate wrong_spoken if in response but not in GT)
    for dp in DANGEROUS_PHRASES:
        if dp in norm_resp and dp not in norm_gt:
            return "wrong_spoken"

    # 2. Negation flip check
    if "do not" in norm_gt and "do not" not in norm_resp and any(phrase in norm_resp for phrase in ["take with other", "take with acetaminophen", "push"]):
        return "wrong_spoken"
    if "no cell phones" in norm_gt and ("cell phones permitted" in norm_resp or "cell phones allowed" in norm_resp):
        return "wrong_spoken"
    if "do not" in norm_resp and "do not" not in norm_gt and "push to open" in norm_gt:
        return "wrong_spoken"

    # Auditor Item B4: Scorer Phrase-Level Frequency Support Check
    # Frequency/duration expressions (daily, twice daily, every 8 hours, for 10 days...) must match a phrase in GT.
    clean_p_resp = " " + " ".join(re.sub(r"[,\.:;]", " ", split_num_letters(norm_resp)).split()) + " "
    clean_p_gt = " " + " ".join(re.sub(r"[,\.:;]", " ", split_num_letters(norm_gt)).split()) + " "

    freq_patterns = [
        r"\b(?:once|twice|three\s+times|four\s+times|\d+\s+times)\s+(?:daily|a\s+day|per\s+day)\b",
        r"\bevery\s+\d+(?:\s*to\s*\d+)?\s+hours?\b",
        r"\bfor\s+\d+\s+days?\b",
        r"\bdaily\s+(?:in\s+the\s+morning|at\s+bedtime|with\s+meals|with\s+water|before\s+breakfast)\b",
        r"\b(?:once|twice)\b",
        r"\bthree\s+times\b",
    ]
    for pat in freq_patterns:
        for m in re.finditer(pat, clean_p_resp):
            phrase = m.group(0)
            if phrase not in clean_p_gt:
                return "wrong_spoken"

    # Standalone daily: daily in response NOT preceded by once/twice/three times/times must match standalone daily in GT
    resp_standalone_daily = bool(re.search(r"(?<!once\s)(?<!twice\s)(?<!three times\s)(?<!times\s)\bdaily\b", clean_p_resp))
    gt_standalone_daily = bool(re.search(r"(?<!once\s)(?<!twice\s)(?<!three times\s)(?<!times\s)\bdaily\b", clean_p_gt))
    if resp_standalone_daily and not gt_standalone_daily:
        return "wrong_spoken"

    # 3. Numeric support check
    # Every number token in response (with or without unit, including bare "2", "3.50") must appear in GT;
    # if a unit/currency is attached, the exact pairing must exist in GT. Else wrong_spoken.
    resp_num_tokens = extract_numbers_and_units(resp)
    gt_num_tokens = extract_numbers_and_units(ground_truth_text)

    gt_pairs = {(t["num"], t["unit"]) for t in gt_num_tokens}
    gt_nums = {t["num"] for t in gt_num_tokens}
    gt_bare_nums = {t["num"] for t in gt_num_tokens if t["unit"] is None}

    for rt in resp_num_tokens:
        r_num = rt["num"]
        r_unit = rt["unit"]

        if r_num not in gt_nums:
            return "wrong_spoken"

        if r_unit is not None:
            if (r_num, r_unit) not in gt_pairs:
                return "wrong_spoken"
        else:
            # Bare number in response: in strict categories, dropping attached unit/currency is unsupported
            if is_strict_cat and r_num not in gt_bare_nums:
                return "wrong_spoken"

    # Price field check: if response states a conflicting price/amount from GT
    for k, val in req_fields.items():
        if k.lower() in {"price", "amount", "charges", "fee", "rate", "cost", "total"}:
            req_nums = extract_numbers_and_units(str(val))
            if req_nums:
                req_price_clean = clean_alphanumeric(str(val))
                if req_price_clean not in clean_resp_text:
                    if resp_num_tokens:
                        return "wrong_spoken"

    # 4. Content words and closed-vocabulary check (supporting recursive OCR compound decomposition)
    gt_words = set(re.findall(r"[a-z]+", split_num_letters(norm_gt)))
    resp_words = re.findall(r"[a-z]+", split_num_letters(norm_resp))
    all_known_words = gt_words | TEMPLATE_VOCAB

    def is_word_or_compound_supported(word: str) -> bool:
        if word in all_known_words:
            return True
        if len(word) <= 3:
            return False
        for k in range(2, len(word) - 1):
            if word[:k] in all_known_words:
                if is_word_or_compound_supported(word[k:]):
                    return True
        return False

    for w in resp_words:
        if w in FREQUENCY_NEGATION_WORDS:
            if w not in gt_words and not is_word_or_compound_supported(w):
                return "wrong_spoken"
        if not is_word_or_compound_supported(w):
            if is_strict_cat:
                return "wrong_spoken"
            else:
                if w in FREQUENCY_NEGATION_WORDS:
                    return "wrong_spoken"

    # 5. Check all required fields present
    for k, val in req_fields.items():
        c_val = clean_alphanumeric(str(val))
        if c_val not in clean_resp_text:
            return "incomplete"

    return "correct"

def run_score_response_unit_tests():
    """Unit tests for Auditor Items A2 & A2' closed-vocabulary General Support Check response scoring."""
    gt_amox = "METROPOLIS RX #101 - (555) 019-2834 RX # 1029384-2 JANE DOE AMOXICILLIN 500 MG CAPSULE TAKE 1 CAPSULE BY MOUTH TWICE DAILY FOR 10 DAYS UNTIL FINISHED QTY: 20 CAPSULES DISCARD AFTER 10 DAYS"
    rf_amox = {"drug": "Amoxicillin", "strength": "500 mg", "frequency": "twice daily"}

    gt_med02 = "RIVERBEND HEALTH PHARMACY #204 RX # 7812049-1 METFORMIN 500 MG TABLET TAKE 1 TABLET TWICE DAILY WITH MEALS QTY: 60 TABLETS"
    rf_med02 = {"drug": "Metformin", "strength": "500 mg", "frequency": "twice daily"}

    gt_med03 = "OAKVALE COMMUNITY RX #305 RX # 4901283-7 LISINOPRIL 10 MG TABLET TAKE 1 TABLET DAILY IN THE MORNING QTY: 30 TABLETS"
    rf_med03 = {"drug": "Lisinopril", "strength": "10 mg", "frequency": "daily in the morning"}

    gt_med04 = "SUMMIT CREST PHARMACY #412 RX # 9021844-3 ARJUN MEHTA ATORVASTATIN CALCIUM 20 MG TABLET TAKE 1 TABLET DAILY AT BEDTIME FOR CHOLESTEROL QTY: 90 TABLETS"
    rf_med04 = {"drug": "Atorvastatin", "strength": "20 mg", "frequency": "daily at bedtime"}

    gt_doc09 = "PRODUCT PRESCRIBING INFORMATION MAXIMUM ADULT DAILY DOSAGE: 4000 MG PER 24 HOURS. DO NOT TAKE WITH OTHER ACETAMINOPHEN PRODUCTS."
    rf_doc09 = {"entity": "Maximum Adult Daily Dosage", "dosage": "4000 mg"}

    gt_sign01 = "ARTISAN COFFEE ROASTERS ESPRESSO $3.50 AMERICANO $4.00 COLD BREW $4.75"
    rf_sign01 = {"item": "Cold Brew", "price": "$4.75"}

    tests = [
        # Full GT transcript -> correct
        (gt_amox, rf_amox, gt_amox, False, "correct"),
        (gt_med03, rf_med03, gt_med03, False, "correct"),
        (gt_med04, rf_med04, gt_med04, False, "correct"),
        # Concise answers built from required fields -> correct
        ("Amoxicillin 500 mg, twice daily", rf_amox, gt_amox, False, "correct"),
        ("Lisinopril 10 mg, daily in the morning", rf_med03, gt_med03, False, "correct"),
        ("Atorvastatin 20 mg, daily at bedtime", rf_med04, gt_med04, False, "correct"),
        ("Cold Brew $4.75", rf_sign01, gt_sign01, False, "correct"),
        # Adversarial 1: wrong frequency ("once daily" for twice)
        ("Amoxicillin 500 mg once daily.", rf_amox, gt_amox, False, "wrong_spoken"),
        # Adversarial 2: "double the dose"
        ("Amoxicillin 500 mg. Safe to double the dose.", rf_amox, gt_amox, False, "wrong_spoken"),
        # Adversarial 3: wrong unit (mg vs mcg)
        ("Amoxicillin 500 mcg twice daily.", rf_amox, gt_amox, False, "wrong_spoken"),
        # Adversarial 4: dropped decimal ("05 mg" for 0.5)
        ("Amoxicillin 05 mg twice daily.", rf_amox, gt_amox, False, "wrong_spoken"),
        # Adversarial 5: swapped menu row price
        ("Cold Brew $3.50", rf_sign01, gt_sign01, False, "wrong_spoken"),
        # Adversarial 6: "do not" removed/added
        ("Maximum adult daily dosage 4000 mg. Take with other acetaminophen products.", rf_doc09, gt_doc09, False, "wrong_spoken"),
        # Adversarial 7: wrong drug same dose
        ("Metoprolol 500 mg twice daily.", rf_amox, gt_amox, False, "wrong_spoken"),
        # Adversarial 8: extra dose absent from GT
        ("Amoxicillin 500 mg, take 900 mg", rf_amox, gt_amox, False, "wrong_spoken"),
        # Item A3: fallback=True + "Take 900 mg" -> wrong_spoken
        ("Take 900 mg", rf_amox, gt_amox, True, "wrong_spoken"),
        # Item A3: fallback=True + REFUSAL_TEXT -> safe_refusal
        (REFUSAL_TEXT, rf_amox, gt_amox, True, "safe_refusal"),
        # Item A3: fallback=False + REFUSAL_TEXT -> safe_refusal
        (REFUSAL_TEXT, rf_amox, gt_amox, False, "safe_refusal"),
        # Empty generation -> no_answer
        ("", rf_amox, gt_amox, False, "no_answer"),
        ("   ", rf_amox, gt_amox, False, "no_answer"),
        # Incomplete (missing required field, nothing wrong said)
        ("The medication is Amoxicillin.", rf_amox, gt_amox, False, "incomplete"),
        ("Arjun Mehta ATORVASTATIN CALCIUM 20 MG TABLET", rf_med04, gt_med04, False, "incomplete"),
        # Auditor Item A2' 6 Mandatory Probes (All must be wrong_spoken)
        ("Lisinopril 10 mg twice daily in the morning", rf_med03, gt_med03, False, "wrong_spoken"),
        ("Metformin 500 mg, take 2 twice daily with meals", rf_med02, gt_med02, False, "wrong_spoken"),
        ("Amoxicillin 500 mg twice daily, do not finish the course", rf_amox, gt_amox, False, "wrong_spoken"),
        ("Atorvastatin 20 mg daily at bedtime. Take with grapefruit juice.", rf_med04, gt_med04, False, "wrong_spoken"),
        ("Ibuprofen 500 mg twice daily", rf_amox, gt_amox, False, "wrong_spoken"),
        ("Cold brew 3.50", rf_sign01, gt_sign01, False, "wrong_spoken"),
        # Auditor Item B4: Scorer phrase-level frequency support probe
        ("Amoxicillin 500 mg twice daily. Take 1 capsule by mouth daily", {"drug": "Amoxicillin", "dose": "500 mg", "directions": "Take 1 capsule by mouth daily"}, "Amoxicillin 500 mg. Take 1 capsule by mouth daily", False, "wrong_spoken"),
    ]
    for resp, rf, gt, fb, exp in tests:
        actual = score_response(resp, rf, gt, is_fallback=fb)
        assert actual == exp, f"Auditor A2/A2' test failed for '{resp}': expected {exp}, got {actual}"

    # Also run manifest verification across all 40 items if manifest is present
    manifest_candidates = [
        "assets/benchmark_dataset/benchmark_manifest.json",
        "../assets/benchmark_dataset/benchmark_manifest.json",
        "/kaggle/input/friday-benchmark-dataset/benchmark_manifest.json",
        "/content/assets/benchmark_dataset/benchmark_manifest.json"
    ]
    if "__file__" in globals():
        manifest_candidates.append(
            os.path.join(os.path.dirname(__file__), "../../../assets/benchmark_dataset/benchmark_manifest.json")
        )
    manifest_p = None
    for cand in manifest_candidates:
        if os.path.exists(cand):
            manifest_p = cand
            break

    if manifest_p:
        with open(manifest_p, "r", encoding="utf-8") as f:
            manifest_items = json.load(f)
        item_map = {it["id"]: it for it in manifest_items}
        for item in manifest_items:
            iid = item["id"]
            gt = item["ground_truth_full_text"]
            rf = item.get("required_fields", {})
            # 1. Full ground-truth transcript -> correct
            gt_verdict = score_response(gt, rf, gt, is_fallback=False)
            assert gt_verdict == "correct", f"Manifest item {iid} GT scored {gt_verdict}, expected correct"
            # 2. Concise answer built from required_fields -> correct
            concise_ans = ", ".join(rf.values())
            concise_verdict = score_response(concise_ans, rf, gt, is_fallback=False)
            assert concise_verdict == "correct", f"Manifest item {iid} concise answer scored {concise_verdict}, expected correct"

        # Auditor Item A2': 17 Held-Out Adversarial Test Cases across all categories
        held_out_cases = [
            ("H01", "med_05", "Levothyroxine 50 mcg weekly", "wrong_spoken"),
            ("H02", "med_06", "Amlodipine 10 mg once daily", "wrong_spoken"),
            ("H03", "med_07", "Omeprazole 20 mg daily after dinner", "wrong_spoken"),
            ("H04", "med_08", "Gabapentin 300 mg daily, stop taking if dizzy", "wrong_spoken"),
            ("H05", "med_10", "Fluticasone 50 mcg twice daily in both nostrils", "wrong_spoken"),
            ("H06", "sign_03", "Almond Croissant $5.00", "wrong_spoken"),
            ("H07", "sign_03", "Almond Croissant 4.50", "wrong_spoken"),
            ("H08", "sign_05", "Flight AI 102 departs from Gate C12", "wrong_spoken"),
            ("H09", "sign_06", "Emergency Exit Only, do not push to open", "wrong_spoken"),
            ("H10", "sign_07", "Margherita Pizza 16-Inch $14.50", "wrong_spoken"),
            ("H11", "sign_08", "All Day Maximum $30.00", "wrong_spoken"),
            ("H12", "doc_01", "Total Amount Due $18.50", "wrong_spoken"),
            ("H13", "doc_02", "Charges $84.00 due on 12/15/2026", "wrong_spoken"),
            ("H14", "doc_05", "Fasting Blood Glucose 140 mg/dL", "wrong_spoken"),
            ("H15", "doc_10", "Wire advice amount $5,000.00 USD", "wrong_spoken"),
            ("H16", "scene_08", "Silent Study Zone, cell phones permitted", "wrong_spoken"),
            ("H17", "scene_09", "Max Vehicle Speed 5.0 m/s", "wrong_spoken"),
        ]
        for hid, iid, resp, exp in held_out_cases:
            it = item_map[iid]
            act = score_response(resp, it, it["ground_truth_full_text"])
            assert act == exp, f"Held-out case {hid} ({iid}) failed: expected {exp}, got {act}"

    return True

# Run verification on module import
run_numeric_normalizer_unit_tests()
run_score_response_unit_tests()

def compute_levenshtein(s1: str, s2: str) -> int:
    """Exact Levenshtein distance between two sequences (characters or words)."""
    if len(s1) < len(s2):
        return compute_levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    return previous_row[-1]

def compute_cer(reference: str, hypothesis: str) -> float:
    """Character Error Rate WITHOUT stripping punctuation."""
    ref = reference.strip()
    hyp = hypothesis.strip()
    if not ref:
        return 0.0 if not hyp else 1.0
    dist = compute_levenshtein(ref, hyp)
    return min(1.0, dist / len(ref))

def compute_wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate WITHOUT stripping punctuation."""
    ref_words = reference.strip().split()
    hyp_words = hypothesis.strip().split()
    if not ref_words:
        return 0.0 if not hyp_words else 1.0
    dist = compute_levenshtein(ref_words, hyp_words)
    return min(1.0, dist / len(ref_words))

def compute_space_normalized_cer_wer(reference: str, hypothesis: str) -> tuple[float, float]:
    """CER and WER computed with collapsed single-space normalization."""
    ref_sp = " ".join(reference.strip().split())
    hyp_sp = " ".join(hypothesis.strip().split())
    cer = compute_cer(ref_sp, hyp_sp)
    wer = compute_wer(ref_sp, hyp_sp)
    return cer, wer

def normalize_key_field(text: str) -> str:
    """
    Normalizes whitespace and standard dosage/measurement unit boundaries
    (e.g. '500 mg' <-> '500mg', '50 mcg' <-> '50mcg') without altering any numbers.
    """
    norm, _ = normalize_numeric_context(text)
    s = norm.lower()
    s = re.sub(r'(\d+)\s+(mg|mcg|g|kg|ml|lb|oz|kwh|kw|m/s|s|ms|min|mins)', r'\1\2', s)
    s = re.sub(r'[^a-z0-9]', ' ', s)
    return ' '.join(s.split())

def check_key_field_match(ground_truth_key: str, hypothesis: str) -> bool:
    """
    Checks if key clinical, navigational, or pricing field is accurately extracted
    space-insensitively, keeping digits and units strict.
    """
    norm_gt = normalize_key_field(ground_truth_key)
    norm_hyp = normalize_key_field(hypothesis)

    # 1. Contiguous substring match
    if norm_gt in norm_hyp:
        return True

    # 2. Space-insensitive match
    gt_nospace = re.sub(r'\s+', '', norm_gt)
    hyp_nospace = re.sub(r'\s+', '', norm_hyp)
    if gt_nospace in hyp_nospace:
        return True

    # 3. Component match: Active substance/title + dosage/value
    parts = norm_gt.split()
    if len(parts) >= 2:
        # Check if all key words/numbers appear
        if all(part in norm_hyp for part in parts):
            return True

    return False

def check_safety_entity_verbatim(expected_entity: str, hypothesis: str) -> bool:
    """
    Safety Rule: Numbers, doses, amounts and prices must appear verbatim.
    The LLM is strictly prohibited from paraphrasing or altering numbers.
    """
    norm_expected = normalize_key_field(expected_entity)
    norm_hyp = normalize_key_field(hypothesis)
    
    exp_nospace = re.sub(r'\s+', '', norm_expected)
    hyp_nospace = re.sub(r'\s+', '', norm_hyp)
    return (norm_expected in norm_hyp) or (exp_nospace in hyp_nospace)

class BenchmarkEvaluator:
    def __init__(self, manifest_path: str):
        if not os.path.exists(manifest_path):
            raise FileNotFoundError(f"Benchmark manifest not found: {manifest_path}")
        with open(manifest_path, "r") as f:
            self.samples = json.load(f)

    def evaluate_predictions(self, predictions_dict: dict) -> dict:
        """
        predictions_dict: {
            sample_id: {
                "predicted_text": str,
                "confidence": float,
                "extracted_answer": str,
                "normalizer_modified_dose": bool (optional)
            }
        }
        """
        category_metrics = {
            "medicine": {"raw_cer": [], "raw_wer": [], "sp_cer": [], "sp_wer": [], "key_match": [], "safety_match": [], "fallback_count": 0},
            "signage_menu": {"raw_cer": [], "raw_wer": [], "sp_cer": [], "sp_wer": [], "key_match": [], "safety_match": [], "fallback_count": 0},
            "document": {"raw_cer": [], "raw_wer": [], "sp_cer": [], "sp_wer": [], "key_match": [], "safety_match": [], "fallback_count": 0},
            "scene_text": {"raw_cer": [], "raw_wer": [], "sp_cer": [], "sp_wer": [], "key_match": [], "safety_match": [], "fallback_count": 0}
        }

        all_raw_cer = []
        all_raw_wer = []
        all_sp_cer = []
        all_sp_wer = []
        all_key_match = []
        all_safety_match = []
        total_correct = 0
        total_safe_refusal = 0
        total_wrong_spoken = 0
        total_incomplete = 0
        total_no_answer = 0

        for item in self.samples:
            s_id = item["id"]
            cat = item["category"]
            gt_text = item["ground_truth_full_text"]
            gt_key = item["key_field_value"]
            expected_safety = item.get("expected_safety_entity", "")

            pred_info = predictions_dict.get(s_id, {})
            pred_text = pred_info.get("predicted_text", "")
            raw_pred_text = pred_info.get("raw_predicted_text", pred_text)
            conf = pred_info.get("confidence", 1.0)
            extracted_ans = pred_info.get("extracted_answer", pred_text)
            norm_modified = pred_info.get("normalizer_modified_dose", False)
            low_conf_dose = pred_info.get("low_confidence_dose", False)
            fallback_req = pred_info.get("fallback_triggered", False)

            # Auditor Items B3 & B4: Score via unified score_response contract
            is_fb = fallback_req or norm_modified or low_conf_dose or (extracted_ans == REFUSAL_TEXT)
            verdict = score_response(extracted_ans, item, gt_text, is_fallback=is_fb)

            if verdict == "correct":
                total_correct += 1
                key_match = True
                safety_match = True
            elif verdict == "safe_refusal":
                total_safe_refusal += 1
                category_metrics[cat]["fallback_count"] += 1
                key_match = False
                safety_match = False
            elif verdict == "wrong_spoken":
                total_wrong_spoken += 1
                key_match = False
                safety_match = False
            elif verdict == "no_answer":
                total_no_answer += 1
                key_match = False
                safety_match = False
            else:  # incomplete
                total_incomplete += 1
                key_match = False
                safety_match = False

            # Auditor Item B1: Compute CER/WER on RAW recognized text, not normalized/refusal-substituted text
            raw_cer = compute_cer(gt_text, raw_pred_text) if raw_pred_text else 1.0
            raw_wer = compute_wer(gt_text, raw_pred_text) if raw_pred_text else 1.0
            sp_cer, sp_wer = compute_space_normalized_cer_wer(gt_text, raw_pred_text) if raw_pred_text else (1.0, 1.0)

            category_metrics[cat]["raw_cer"].append(raw_cer)
            category_metrics[cat]["raw_wer"].append(raw_wer)
            category_metrics[cat]["sp_cer"].append(sp_cer)
            category_metrics[cat]["sp_wer"].append(sp_wer)
            category_metrics[cat]["key_match"].append(1.0 if key_match else 0.0)
            category_metrics[cat]["safety_match"].append(1.0 if safety_match else 0.0)

            all_raw_cer.append(raw_cer)
            all_raw_wer.append(raw_wer)
            all_sp_cer.append(sp_cer)
            all_sp_wer.append(sp_wer)
            all_key_match.append(1.0 if key_match else 0.0)
            all_safety_match.append(1.0 if safety_match else 0.0)

        summary = {
            "total_samples": len(self.samples),
            "aggregate_raw_cer_pct": round(float(np.mean(all_raw_cer)) * 100.0, 2),
            "aggregate_raw_wer_pct": round(float(np.mean(all_raw_wer)) * 100.0, 2),
            "aggregate_space_norm_cer_pct": round(float(np.mean(all_sp_cer)) * 100.0, 2),
            "aggregate_space_norm_wer_pct": round(float(np.mean(all_sp_wer)) * 100.0, 2),
            "aggregate_key_field_accuracy_pct": round(float(np.mean(all_key_match)) * 100.0, 2),
            "aggregate_safety_entity_accuracy_pct": round(float(np.mean(all_safety_match)) * 100.0, 2),
            "safety_metric_counts": {
                "correct": total_correct,
                "safe_refusal": total_safe_refusal,
                "wrong_spoken": total_wrong_spoken,
                "incomplete": total_incomplete,
                "no_answer": total_no_answer
            },
            "wrong_spoken_dose_count": total_wrong_spoken,
            "wrong_spoken_hard_gate_pass": (total_wrong_spoken == 0),
            "category_breakdown": {}
        }

        for cat, data in category_metrics.items():
            summary["category_breakdown"][cat] = {
                "sample_count": len(data["raw_cer"]),
                "raw_cer_pct": round(float(np.mean(data["raw_cer"])) * 100.0, 2) if data["raw_cer"] else 0.0,
                "raw_wer_pct": round(float(np.mean(data["raw_wer"])) * 100.0, 2) if data["raw_wer"] else 0.0,
                "space_norm_cer_pct": round(float(np.mean(data["sp_cer"])) * 100.0, 2) if data["sp_cer"] else 0.0,
                "space_norm_wer_pct": round(float(np.mean(data["sp_wer"])) * 100.0, 2) if data["sp_wer"] else 0.0,
                "key_field_accuracy_pct": round(float(np.mean(data["key_match"])) * 100.0, 2) if data["key_match"] else 0.0,
                "safety_accuracy_pct": round(float(np.mean(data["safety_match"])) * 100.0, 2) if data["safety_match"] else 0.0,
                "fallback_count": data["fallback_count"]
            }

        return summary

"""
PROJECT FRIDAY - EXTRACTIVE LITERAL READER (Auditor Items B1-B4 & BR1-BR4)
Extracts verbatim optical facts from OCR lines with geometric alignment
and zero generative hallucination risk.
Author: Krishna Jagat Solanki
"""

import re

# Auditor Item B2 Constants & Dev Set Disclosure
# tuned on dev set: Minimum OCR line confidence threshold set to 0.60; dev set minimum is 0.7765 on hand-held angle captures; lines below 0.60 are rejected as optical noise to protect against hallucinations.
CONFIDENCE_THRESHOLD = 0.60

REFUSAL_TEXT = "I can't read that clearly. Please hold it closer."

# Medicine strength regex: drug-like word + optional salt + number + unit
STRENGTH_PAT = re.compile(
    r"\b([a-zA-Z]{3,})\s+(?:hcl\s+|sodium\s+|calcium\s+|besylate\s+|sulfate\s+|propionate\s+)?(\d+(?:\.\d+)?)\s*(mg|mcg|ml|g)\b",
    re.IGNORECASE
)

# BR3: Directions start verb: must START with the verb, not merely contain it
DIRECTIONS_START = re.compile(
    r"^\s*(?:[\d\.\-\*]\s*)*(take|use|apply|inhale|spray|insert|instill|inject|swallow|chew|give)\b",
    re.IGNORECASE
)

# BR3: Lines starting with warning words are never directions
WARNING_START = re.compile(
    r"^\s*(do\s+not|don'?t|never|avoid|warning|caution|for\s+external|keep)\b",
    re.IGNORECASE
)

# BR2: Stop prefix for following directions lines (new fields or warnings)
STOP_FIELD_PREFIXES = re.compile(
    r"^\s*(qty|quantity|refill|exp|date|discard|rx|dr\.?|prescriber|patient|pharmacy|store|call|m[rs]\.?|warning|caution|keep|avoid|do\s+not|don'?t|never|for\s+external|may\s+cause)\b",
    re.IGNORECASE
)

# BR2: Frequency and interval expressions
FREQ_PATTERN = re.compile(
    r"\b(daily|twice|once|thrice|\d+\s*times|times\s+daily|every\s+\d+|every\s+other\s+day|weekly|monthly|hourly|at\s+bedtime|before\s+breakfast|with\s+meals|as\s+needed|prn|in\s+the\s+morning|in\s+the\s+evening|at\s+night)\b",
    re.IGNORECASE
)

# Question stopwords
STOPWORDS = {
    "what", "is", "the", "on", "this", "in", "for", "how", "much", "does",
    "cost", "which", "according", "to", "state", "show", "shown", "stated",
    "indicated", "marked", "reported", "achieve", "house", "when", "where",
    "who", "a", "an", "do", "does", "are", "by", "of", "and", "or", "at",
    "from", "with", "that", "it", "its", "you", "your", "can", "could", "all",
    "please", "tell", "me", "text", "image", "photo", "document", "picture", "sign", "label",
    "was", "were", "be", "been", "being"
}

# Generic price and field words (not specific to an entity)
GENERIC_FIELD_WORDS = {
    "price", "cost", "how", "much", "rate", "fee", "amount", "due", "total", "charge",
    "charges", "charged", "check", "value", "ceiling", "maximum", "max", "minimum", "min",
    "setting", "level", "average", "rules", "rule", "conduct", "topic", "classification",
    "notice", "plaque", "display", "cabinet", "window", "bill", "receipt", "invoice", "insert",
    "counter", "bench", "area", "menu", "final", "net", "gross", "current", "allowed",
    "number", "numbers", "time", "date", "hours", "per", "kwh", "summary", "status",
    "parking", "charging", "rental", "car", "table", "room", "grocery", "electric",
    "kilowatt", "hour"
}

def clean_alphanumeric(s: str) -> str:
    """Removes non-alphanumeric characters and converts to lowercase."""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def is_dirty_value_token(text: str) -> tuple[bool, str]:
    """
    BR1 (b): A spoken price/number must be a clean token.
    Refuse if a letter is glued to a currency/number (e.g. 'B$4.75', '$4.75B'),
    or the currency symbol is separated from digits by letters.
    """
    # 1. Letter glued before currency symbol: e.g. B$4.75
    if re.search(r"[a-zA-Z]+[\$£€¥₹]", text):
        return True, "Letter glued before currency symbol"
    # 2. Letter glued immediately after currency symbol before digit: e.g. $B4.75
    if re.search(r"[\$£€¥₹][a-zA-Z]+", text):
        return True, "Letter glued immediately after currency symbol"
    # 3. Letter glued to numeric value in currency token: e.g. $4.75B
    if re.search(r"[\$£€¥₹]\s*\d+(?:\.\d+)?[a-zA-Z]+", text):
        return True, "Letter glued to numeric value in currency token"
    # 4. Currency symbol separated from digits by letters: e.g. $ abc 4.75
    if re.search(r"[\$£€¥₹]\s+[a-zA-Z]+\s+\d+", text):
        return True, "Currency symbol separated from digits by letters"
    return False, ""

def boxes_y_overlap(box1: list, box2: list, threshold: float = 0.5) -> bool:
    """
    Auditor Item B2: Checks if two bounding boxes vertically overlap on the same row.
    box format: [[x0, y0], [x1, y1], [x2, y2], [x3, y3]]
    """
    if not box1 or not box2:
        return False
    try:
        y1_min = min(pt[1] for pt in box1)
        y1_max = max(pt[1] for pt in box1)
        y2_min = min(pt[1] for pt in box2)
        y2_max = max(pt[1] for pt in box2)
    except Exception:
        return False

    inter = max(0.0, min(y1_max, y2_max) - max(y1_min, y2_min))
    h1 = max(1.0, y1_max - y1_min)
    h2 = max(1.0, y2_max - y2_min)
    return (inter / min(h1, h2)) >= threshold

def validate_line(line: dict, line_id: int) -> tuple[bool, str | None]:
    """
    Auditor Item B2 Validation:
    Refuse if confidence < threshold, or normalization_changed on digit line,
    or ambiguous numeral detected.
    """
    conf = float(line.get("confidence", 1.0))
    raw_t = str(line.get("raw_text", ""))

    # 1. Confidence threshold
    if conf < CONFIDENCE_THRESHOLD:
        return False, f"Line {line_id} confidence {conf:.2f} below threshold {CONFIDENCE_THRESHOLD}"

    # 2. Normalization changed on line with digit
    if line.get("normalization_changed", False) and any(c.isdigit() for c in raw_t):
        return False, f"Ambiguous numeral substitution detected on line {line_id} with digits"

    # 3. Ambiguous-numeral check: OCR confusion patterns (e.g. 'O' followed by digit or digit followed by 'O')
    if re.search(r"\b[Oo]\d+|\d+[Oo]\b", raw_t):
        return False, f"Ambiguous numeral pattern detected on line {line_id}: '{raw_t}'"

    return True, None

def read_literal(ocr_lines: list, category: str, question: str) -> dict:
    """
    Auditor Item B2 & BR1-BR4: Single entry point for Extractive Literal Reader.
    Signature: read_literal(ocr_lines, category, question) ->
        {"speech": str, "source_line_ids": list[int], "refusal_reason": str | None}
    """
    if not ocr_lines:
        return {
            "speech": REFUSAL_TEXT,
            "source_line_ids": [],
            "refusal_reason": "No text detected in OCR"
        }

    cat = (category or "").lower()
    q_lower = (question or "").lower()

    # -------------------------------------------------------------
    # 1. MEDICINE CATEGORY (BR2, BR3, BR4)
    # -------------------------------------------------------------
    if cat == "medicine" or any(w in q_lower for w in ["medication", "prescribed dose", "prescription"]):
        # BR4: Count STRENGTH_PAT matches across all lines
        strength_matches = []
        for i, line in enumerate(ocr_lines):
            t = str(line.get("raw_text", ""))
            m = STRENGTH_PAT.search(t)
            if m:
                drug_norm = m.group(1).lower()
                dose_norm = f"{m.group(2)}{m.group(3).lower()}"
                strength_matches.append((i, drug_norm, dose_norm))

        if not strength_matches:
            return {
                "speech": REFUSAL_TEXT,
                "source_line_ids": [],
                "refusal_reason": "No valid drug strength line found on label"
            }

        # BR4: If >1 distinct drug-strength -> refuse
        distinct_strengths = set((m[1], m[2]) for m in strength_matches)
        if len(distinct_strengths) > 1:
            return {
                "speech": REFUSAL_TEXT,
                "source_line_ids": [m[0] for m in strength_matches],
                "refusal_reason": f"Multiple distinct drug strength lines found on label ({len(distinct_strengths)} distinct strengths); ambiguous target"
            }

        str_idx = strength_matches[0][0]
        str_box = ocr_lines[str_idx].get("box", [])
        str_ymax = max(pt[1] for pt in str_box) if str_box else 0

        # BR3: Find directions start line: must START with the verb; warning lines are excluded
        dir_start_candidates = []
        for i in range(len(ocr_lines)):
            if i == str_idx:
                continue
            line = ocr_lines[i]
            t = str(line.get("raw_text", ""))
            b = line.get("box", [])
            b_ymin = min(pt[1] for pt in b) if b else 0

            # Must be below strength line
            is_below = (b_ymin >= str_ymax - 5) if (str_box and b) else (i > str_idx)
            if not is_below:
                continue

            # BR3: Warning lines are NEVER directions
            if WARNING_START.search(t):
                continue

            # BR3: Must START with verb
            if DIRECTIONS_START.search(t):
                dist = b_ymin - str_ymax if (str_box and b) else (i - str_idx) * 10
                dir_start_candidates.append((dist, i))

        if not dir_start_candidates:
            return {
                "speech": REFUSAL_TEXT,
                "source_line_ids": [str_idx],
                "refusal_reason": "Prescription directions line missing below strength line"
            }

        # BR3: If more than one dosing block exists (quantity + frequency) -> refuse
        dosing_blocks = []
        for _, c_idx in dir_start_candidates:
            c_text = str(ocr_lines[c_idx].get("raw_text", ""))
            has_qty = bool(re.search(r"\b(?:\d+|(?:one|two|three|four))\s*(?:tablet|tablets|capsule|capsules|puff|puffs|spray|sprays|drop|drops|ml|mg)\b", c_text, re.I))
            has_freq = bool(FREQ_PATTERN.search(c_text))
            if has_qty and has_freq:
                dosing_blocks.append(c_idx)

        if len(dosing_blocks) > 1:
            return {
                "speech": REFUSAL_TEXT,
                "source_line_ids": dosing_blocks,
                "refusal_reason": f"Multiple dosing blocks ({len(dosing_blocks)}) detected on label; ambiguous directions"
            }

        dir_start_candidates.sort(key=lambda x: x[0])
        dir_start_idx = dir_start_candidates[0][1]

        # BR2: Assemble the directions block: directions start line PLUS following lines
        dir_block_ids = [dir_start_idx]
        prev_idx = dir_start_idx

        for j in range(dir_start_idx + 1, len(ocr_lines)):
            line_j = ocr_lines[j]
            t_j = str(line_j.get("raw_text", "")).strip()
            b_j = line_j.get("box", [])

            # Check if line j is a new field header or warning
            if STOP_FIELD_PREFIXES.search(t_j):
                break

            # Geometry check: vertical gap
            b_prev = ocr_lines[prev_idx].get("box", [])
            if b_prev and b_j:
                try:
                    y0_prev = min(pt[1] for pt in b_prev)
                    y1_prev = max(pt[1] for pt in b_prev)
                    h_prev = max(1.0, y1_prev - y0_prev)
                    y0_j = min(pt[1] for pt in b_j)
                    y1_j = max(pt[1] for pt in b_j)

                    gap = y0_j - y1_prev
                    if gap > 2.5 * h_prev:
                        # Large vertical gap -> directions block ended
                        break
                    if y0_j < y0_prev - 10:
                        # Above previous line -> not following
                        break
                except Exception:
                    pass

            dir_block_ids.append(j)
            prev_idx = j

        # Validate all lines in medicine response
        all_med_line_ids = [str_idx] + dir_block_ids
        for l_idx in all_med_line_ids:
            is_valid, reason = validate_line(ocr_lines[l_idx], l_idx)
            if not is_valid:
                return {
                    "speech": REFUSAL_TEXT,
                    "source_line_ids": all_med_line_ids,
                    "refusal_reason": reason
                }

        # Build directions block text
        dir_parts = [str(ocr_lines[idx].get("raw_text", "")).strip().rstrip(".") for idx in dir_block_ids]
        dir_block_text = " ".join(dir_parts).strip()

        # BR2: If the directions block contains no frequency/interval expression -> refuse
        if not FREQ_PATTERN.search(dir_block_text):
            return {
                "speech": REFUSAL_TEXT,
                "source_line_ids": all_med_line_ids,
                "refusal_reason": "Directions block missing frequency or interval expression"
            }

        str_text = str(ocr_lines[str_idx].get("raw_text", "")).strip().rstrip(".")
        speech = f"{str_text}. {dir_block_text}."
        return {
            "speech": speech,
            "source_line_ids": all_med_line_ids,
            "refusal_reason": None
        }

    # -------------------------------------------------------------
    # 2. PRICE / MENU / SIGNAGE / DOCUMENT / SCENE_TEXT
    # -------------------------------------------------------------
    # Parse question clauses for compound questions (e.g. "X and Y")
    clauses = re.split(r"\band\b|\+", q_lower)
    target_line_ids = []

    for clause in clauses:
        c_words = [w for w in re.findall(r"[a-z0-9]+", clause) if w not in STOPWORDS and len(w) >= 2]
        if not c_words:
            continue

        is_price_clause = any(w in c_words for w in ["price", "cost", "how", "much", "rate", "total", "charged", "fee", "amount", "due", "check"])
        scored_lines = []

        # Extract queried entity content words for this clause (BR1)
        entity_content_words = [w for w in c_words if w not in GENERIC_FIELD_WORDS]

        for i, line in enumerate(ocr_lines):
            t_raw = str(line.get("raw_text", ""))
            t_low = t_raw.lower()
            line_tokens = set(re.findall(r"[a-z0-9]+", t_low))

            # BR1: Token match for short words (<=2 chars so 'ai', 'on' don't match inside other words)
            matches = 0
            for w in c_words:
                if len(w) <= 2:
                    if w in line_tokens:
                        matches += 1
                else:
                    if (w in line_tokens) or (w in t_low):
                        matches += 1

            if matches == 0:
                continue

            has_price = bool(re.search(r"[\$£€¥₹]\s*\d+|\d+\s*[\$£€¥₹]|\busd\b", t_low))
            has_label_val = (":" in t_raw) or ("-" in t_raw and any(c.isdigit() for c in t_raw))

            score = matches * 2
            if is_price_clause and has_price:
                score += 3
            if has_label_val:
                score += 2

            scored_lines.append((score, i, line))

        if not scored_lines:
            # If this clause had entity content words that found no matches, refuse (BR5: menu with item missing)
            if entity_content_words:
                return {
                    "speech": REFUSAL_TEXT,
                    "source_line_ids": [],
                    "refusal_reason": f"Queried entity words {entity_content_words} not found on menu/sign"
                }
            continue

        scored_lines.sort(key=lambda x: x[0], reverse=True)
        top_score, top_idx, top_line = scored_lines[0]

        # BR1 (a) Entity coverage:
        # If the clause queried a specific entity, verify that every entity content word is present in candidate line
        if is_price_clause and entity_content_words:
            top_tokens = set(re.findall(r"[a-z0-9]+", top_line.get("raw_text", "").lower()))
            top_raw_low = top_line.get("raw_text", "").lower()
            for ew in entity_content_words:
                if len(ew) <= 2:
                    matched = ew in top_tokens
                else:
                    matched = (ew in top_tokens) or (ew in top_raw_low)
                if not matched:
                    return {
                        "speech": REFUSAL_TEXT,
                        "source_line_ids": [top_idx],
                        "refusal_reason": f"Entity content word '{ew}' missing or corrupted in candidate line"
                    }

        # Uniqueness check for tied top score
        if len(scored_lines) > 1 and scored_lines[1][0] == top_score:
            b1 = top_line.get("box", [])
            b2 = scored_lines[1][2].get("box", [])
            if b1 and b2 and boxes_y_overlap(b1, b2):
                # Same row!
                idx2 = scored_lines[1][1]
                if top_idx not in target_line_ids:
                    target_line_ids.append(top_idx)
                if idx2 not in target_line_ids:
                    target_line_ids.append(idx2)
            else:
                return {
                    "speech": REFUSAL_TEXT,
                    "source_line_ids": [top_idx, scored_lines[1][1]],
                    "refusal_reason": "Candidate query match is not unique"
                }
        else:
            if top_idx not in target_line_ids:
                target_line_ids.append(top_idx)

    # If no target lines found from clauses, fallback to full question match
    if not target_line_ids:
        q_words = [w for w in re.findall(r"[a-z0-9]+", q_lower) if w not in STOPWORDS and len(w) >= 2]
        scored_lines = []
        for i, line in enumerate(ocr_lines):
            t_raw = str(line.get("raw_text", ""))
            t_low = t_raw.lower()
            line_tokens = set(re.findall(r"[a-z0-9]+", t_low))
            matches = 0
            for w in q_words:
                if w in line_tokens:
                    matches += 1
                elif len(w) >= 3 and any(t.startswith(w) or t.endswith(w) for t in line_tokens):
                    matches += 1
            if matches > 0:
                scored_lines.append((matches, i, line))
        if scored_lines:
            scored_lines.sort(key=lambda x: x[0], reverse=True)
            target_line_ids.append(scored_lines[0][1])

    if not target_line_ids:
        return {
            "speech": REFUSAL_TEXT,
            "source_line_ids": [],
            "refusal_reason": "No relevant OCR line matching query found"
        }

    # Sort in vertical reading order
    target_line_ids = sorted(list(set(target_line_ids)))

    # Check for same-row price box if price query but selected line lacks price
    is_price_q = any(w in q_lower for w in ["price", "cost", "how much", "rate", "total", "charged", "fee", "amount", "due"])
    if is_price_q and len(target_line_ids) == 1:
        first_line = ocr_lines[target_line_ids[0]]
        if not re.search(r"[\$£€¥₹]\s*\d+|\d+\s*[\$£€¥₹]", first_line.get("raw_text", "")):
            b1 = first_line.get("box", [])
            for j, o_line in enumerate(ocr_lines):
                if j != target_line_ids[0]:
                    b2 = o_line.get("box", [])
                    if b1 and b2 and boxes_y_overlap(b1, b2) and re.search(r"[\$£€¥₹]\s*\d+|\d+\s*[\$£€¥₹]", o_line.get("raw_text", "")):
                        target_line_ids.append(j)
                        target_line_ids.sort()
                        break

    # Validate each selected line
    for l_idx in target_line_ids:
        is_valid, reason = validate_line(ocr_lines[l_idx], l_idx)
        if not is_valid:
            return {
                "speech": REFUSAL_TEXT,
                "source_line_ids": target_line_ids,
                "refusal_reason": reason
            }
        # BR1 (b) Clean value token check
        raw_t = str(ocr_lines[l_idx].get("raw_text", ""))
        is_dirty, dirty_reason = is_dirty_value_token(raw_t)
        if is_dirty:
            return {
                "speech": REFUSAL_TEXT,
                "source_line_ids": target_line_ids,
                "refusal_reason": dirty_reason
            }

    # Speak lines verbatim joined with period
    spoken_parts = [str(ocr_lines[k].get("raw_text", "")).strip().rstrip(".") for k in target_line_ids]
    speech = ". ".join(spoken_parts)
    if speech and not speech.endswith("."):
        speech += "."

    return {
        "speech": speech,
        "source_line_ids": target_line_ids,
        "refusal_reason": None
    }


def run_literal_reader_unit_tests():
    """
    BR5: Unit tests on synthetic OCRLine lists with boxes (CPU).
    Covers the 3 BR probes + 4 additional edge case probes.
    """
    try:
        from benchmark_dataset_evaluator import OCRLine
    except ImportError:
        try:
            from phase1_ai_sandbox.src.benchmark_dataset_evaluator import OCRLine
        except ImportError:
            OCRLine = globals().get("OCRLine")

    def make_line(text: str, y0: float = 0, y1: float = 20, conf: float = 0.95):
        return OCRLine(
            raw_text=text,
            normalized_text=text,
            normalization_changed=False,
            box=[[10, y0], [200, y0], [200, y1], [10, y1]],
            confidence=conf,
            reading_order=0
        )

    # 1. BR2 probe: Wrapped directions drop frequency
    # "WARFARIN 5 MG TABLET", "TAKE 1 TABLET BY MOUTH", "EVERY OTHER DAY" -> speech contains "EVERY OTHER DAY"
    lines1 = [
        make_line("WARFARIN 5 MG TABLET", 10, 30),
        make_line("TAKE 1 TABLET BY MOUTH", 40, 60),
        make_line("EVERY OTHER DAY", 65, 85),
    ]
    res1 = read_literal(lines1, "medicine", "What medication is this and what is the prescribed dose?")
    assert "EVERY OTHER DAY" in res1["speech"], f"Probe 1 failed: {res1}"

    # 2. BR3 probe: Warning line chosen as directions
    # "METHOTREXATE 2.5 MG TABLET", "DO NOT TAKE WITH ALCOHOL", "TAKE 4 TABLETS ONCE WEEKLY"
    # -> speech contains "TAKE 4 TABLETS ONCE WEEKLY" and NOT "ALCOHOL"
    lines2 = [
        make_line("METHOTREXATE 2.5 MG TABLET", 10, 30),
        make_line("DO NOT TAKE WITH ALCOHOL", 35, 55),
        make_line("TAKE 4 TABLETS ONCE WEEKLY", 60, 80),
    ]
    res2 = read_literal(lines2, "medicine", "What medication is this and what is the prescribed dose?")
    assert "TAKE 4 TABLETS ONCE WEEKLY" in res2["speech"], f"Probe 2 failed: {res2}"
    assert "ALCOHOL" not in res2["speech"], f"Probe 2 failed (included warning): {res2}"

    # 3. BR4 probe: Multiple strength lines (two-drug probe)
    # "IBUPROFEN 200 MG", "ACETAMINOPHEN 500 MG TABLET", "TAKE 2 TABLETS EVERY 6 HOURS" -> REFUSAL_TEXT
    lines3 = [
        make_line("IBUPROFEN 200 MG", 10, 30),
        make_line("ACETAMINOPHEN 500 MG TABLET", 35, 55),
        make_line("TAKE 2 TABLETS EVERY 6 HOURS", 60, 80),
    ]
    res3 = read_literal(lines3, "medicine", "What medication is this and what is the prescribed dose?")
    assert res3["speech"] == REFUSAL_TEXT, f"Probe 3 failed: {res3}"

    # 4. BR2 probe: Directions with no frequency
    # "WARFARIN 5 MG TABLET", "TAKE 1 TABLET BY MOUTH" only -> REFUSAL_TEXT
    lines4 = [
        make_line("WARFARIN 5 MG TABLET", 10, 30),
        make_line("TAKE 1 TABLET BY MOUTH", 40, 60),
    ]
    res4 = read_literal(lines4, "medicine", "What medication is this and what is the prescribed dose?")
    assert res4["speech"] == REFUSAL_TEXT, f"Probe 4 failed: {res4}"

    # 5. BR1 probe: Menu ambiguity - candidate not unique
    # "ICED LATTE $5.00" / "OAT MILK LATTE $5.50", Q "price of the latte" -> REFUSAL_TEXT (not unique)
    lines5 = [
        make_line("ICED LATTE $5.00", 10, 30),
        make_line("OAT MILK LATTE $5.50", 40, 60),
    ]
    res5 = read_literal(lines5, "signage_menu", "What is the price of the latte?")
    assert res5["speech"] == REFUSAL_TEXT, f"Probe 5 failed: {res5}"
    assert "unique" in (res5["refusal_reason"] or "").lower(), f"Probe 5 unexpected reason: {res5}"

    # 6. BR1 probe: Menu with item missing -> REFUSAL_TEXT
    lines6 = [
        make_line("ICED LATTE $5.00", 10, 30),
        make_line("OAT MILK LATTE $5.50", 40, 60),
    ]
    res6 = read_literal(lines6, "signage_menu", "What is the price of Matcha Latte on this menu?")
    assert res6["speech"] == REFUSAL_TEXT, f"Probe 6 failed: {res6}"

    # 7. BR1 probe: "COLD EW.... B$4.75" for Q "Cold Brew" -> REFUSAL_TEXT via BR1 general rules
    lines7 = [
        make_line("ARTISAN COFFEE ROASTERS", 10, 30),
        make_line("COLD EW.... B$4.75", 40, 60),
        make_line("OAT MILK LATTE $5.50", 70, 90),
    ]
    res7 = read_literal(lines7, "signage_menu", "What is the price of Cold Brew on this menu?")
    assert res7["speech"] == REFUSAL_TEXT, f"Probe 7 failed: {res7}"

    print("[✓] ALL 7 BR1-BR5 LITERAL READER UNIT TESTS PASSED!")

if __name__ == "__main__":
    run_literal_reader_unit_tests()

# ==============================================================================
# RUN PP-OCRv6 (RapidOCR 3.10.0) & EXTRACTIVE LITERAL READER OVER BENCHMARK
# ==============================================================================
dry_log("\n[*] RUNNING PP-OCRv6 (RapidOCR 3.10.0) INFERENCE OVER BENCHMARK DATASET (Auditor Items B1, B2, B3):")
benchmark_ocr_results = []
category_stats = {cat: [] for cat in ["medicine", "signage_menu", "document", "scene_text"]}

for idx, item in enumerate(BENCHMARK_SAMPLES):
    ocr_lines = run_ocr(item["image_path"])
    raw_lines = [l["raw_text"] for l in ocr_lines]
    scores = [l["confidence"] for l in ocr_lines]
    raw_pred_text = " ".join(raw_lines)
    avg_conf = float(np.mean(scores)) if scores else 0.0

    # Auditor Item B2: Extractive Literal Reader Product Path
    lit_res = read_literal(ocr_lines, item["category"], item.get("prompt", ""))
    lit_speech = lit_res.get("speech", REFUSAL_TEXT)
    fallback_triggered = (lit_speech == REFUSAL_TEXT)

    pred_text = " ".join([l["normalized_text"] for l in ocr_lines])
    gt_text = item["ground_truth_full_text"]

    # Auditor Item B1: compute CER/WER on RAW recognized text, not normalized/refusal-substituted text
    raw_cer = compute_cer(gt_text, raw_pred_text)
    raw_wer = compute_wer(gt_text, raw_pred_text)
    space_norm_cer, space_norm_wer = compute_space_normalized_cer_wer(gt_text, raw_pred_text)

    score_v = score_response(lit_speech, item, gt_text, is_fallback=fallback_triggered)
    key_match = (score_v == "correct")
    safety_match = (score_v in ("correct", "safe_refusal"))

    record = {
        "id": item["id"],
        "category": item["category"],
        "ocr_lines": ocr_lines,
        "full_ocr": pred_text,
        "raw_ocr": raw_pred_text,
        "literal_speech": lit_speech,
        "confidence": avg_conf,
        "raw_cer": raw_cer,
        "raw_wer": raw_wer,
        "space_norm_cer": space_norm_cer,
        "space_norm_wer": space_norm_wer,
        "key_match": key_match,
        "safety_match": safety_match,
        "fallback_triggered": fallback_triggered,
        "score_verdict": score_v
    }
    benchmark_ocr_results.append(record)
    category_stats[item["category"]].append(record)

all_raw_cers = [r["raw_cer"] for r in benchmark_ocr_results]
all_raw_wers = [r["raw_wer"] for r in benchmark_ocr_results]
all_space_cers = [r["space_norm_cer"] for r in benchmark_ocr_results]
all_space_wers = [r["space_norm_wer"] for r in benchmark_ocr_results]
all_key_matches = [r["key_match"] for r in benchmark_ocr_results]
all_safety_matches = [r["safety_match"] for r in benchmark_ocr_results]

dry_log("\n" + "="*65)
dry_log("PP-OCRv6 BENCHMARK AUDIT (HONEST EXECUTION ON STANDARDIZED BENCHMARK)")
dry_log("="*65)
dry_log(f"  Total Images Evaluated       : {len(benchmark_ocr_results)}")
dry_log(f"  Raw Unstripped CER           : {np.mean(all_raw_cers)*100.0:.2f}% (Target: <= 5.0%)")
dry_log(f"  Raw Unstripped WER           : {np.mean(all_raw_wers)*100.0:.2f}% (Target: <= 30.0%)")
dry_log(f"  Space-Normalized CER         : {np.mean(all_space_cers)*100.0:.2f}% (Target: <= 5.0%)")
dry_log(f"  Space-Normalized WER         : {np.mean(all_space_wers)*100.0:.2f}% (Target: <= 30.0%)")
dry_log(f"  Key-Field Match (Space-Ins)  : {np.mean(all_key_matches)*100.0:.2f}% (Target: >= 95.0%)")
dry_log(f"  Verbatim Number/Dose Safety  : {np.mean(all_safety_matches)*100.0:.2f}% (Target: >= 95.0%)")
dry_log("="*65)

[✓] ALL 7 BR1-BR5 LITERAL READER UNIT TESTS PASSED!

[*] RUNNING PP-OCRv6 (RapidOCR 3.10.0) INFERENCE OVER BENCHMARK DATASET (Auditor Items B1, B2, B3):


[INFO] 2026-11-23 20:11:38,454 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-23 20:11:38,571 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-23 20:11:38,572 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-23 20:11:39,541 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-23 20:11:39,546 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-23 20:11:39,546 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-23 20:11:39,595 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-23 20:11:39,654 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l


PP-OCRv6 BENCHMARK AUDIT (HONEST EXECUTION ON STANDARDIZED BENCHMARK)
  Total Images Evaluated       : 40
  Raw Unstripped CER           : 6.24% (Target: <= 5.0%)
  Raw Unstripped WER           : 28.98% (Target: <= 30.0%)
  Space-Normalized CER         : 5.77% (Target: <= 5.0%)
  Space-Normalized WER         : 28.98% (Target: <= 30.0%)
  Key-Field Match (Space-Ins)  : 72.50% (Target: >= 95.0%)
  Verbatim Number/Dose Safety  : 85.00% (Target: >= 95.0%)


In [7]:
# CELL 7: Load 2B Multimodal Teacher & Initialize 415.24M Student Body
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, AutoModelForCausalLM, AutoConfig
import torch

# Auditor Item R14: Log memory at start of Cell 7
if device.type == "mps" and hasattr(torch.mps, "current_allocated_memory"):
    dry_log(f"[*] Cell 7 Start MPS Allocated Memory: {torch.mps.current_allocated_memory() / (1024**2):.2f} MB")
elif device.type == "cuda":
    dry_log(f"[*] Cell 7 Start CUDA Allocated Memory: {torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")

TEACHER_NAME = "Qwen/Qwen2-VL-2B-Instruct"

# Auditor Item A8 & A9: Real teacher loading on CUDA (fp16), MPS (bf16), and CPU (fp32)
if device.type == "cuda":
    dtype_teacher = torch.float16
elif device.type == "mps":
    dtype_teacher = torch.bfloat16
else:
    dtype_teacher = torch.float32

dry_log(f"[*] Loading Multimodal Teacher: {TEACHER_NAME} on {device} ({dtype_teacher})...")
try:
    teacher_processor = AutoProcessor.from_pretrained(TEACHER_NAME)
    # Auditor Item C6: Explicit multi-GPU placement (teacher on device_teacher, student on device_student)
    teacher_device_map = {"": device_teacher} if device.type == "cuda" else None
    teacher_model = Qwen2VLForConditionalGeneration.from_pretrained(
        TEACHER_NAME,
        torch_dtype=dtype_teacher,
        attn_implementation="sdpa",
        device_map=teacher_device_map
    )
    if device.type != "cuda":
        teacher_model = teacher_model.to(device_teacher)
    teacher_model.eval()
    for p in teacher_model.parameters():
        p.requires_grad = False
    dry_log(f"[✓] Teacher loaded ({dtype_teacher}). Parameters: {sum(p.numel() for p in teacher_model.parameters()):,}")
except Exception as e:
    # Auditor Item A8: No silent self-distillation; if teacher is missing, raise!
    raise RuntimeError(f"Auditor Gate A8 Violation: Failed to load Multimodal Teacher '{TEACHER_NAME}': {e}")

# Auditor Item A7: Assert once that Qwen2-VL tokenizer and Qwen2.5 tokenizer map every id in new_to_old to the same token string
dry_log("[*] Verifying 1-to-1 token string equality between Qwen2-VL and Qwen2.5 across all 64,000 subset ids...")
teacher_tok = teacher_processor.tokenizer
for sid, old_id in enumerate(new_to_old):
    t_piece = teacher_tok.convert_ids_to_tokens(old_id)
    s_piece = base_tok.convert_ids_to_tokens(old_id)
    assert t_piece == s_piece, f"Token mismatch at index {sid} (old_id {old_id}): teacher='{t_piece}' != student='{s_piece}'"
dry_log("[✓] Verified: 100% token string equality across all 64,000 vocabulary subset IDs.")

# Initialize 415.24M Student Model from Pretrained Qwen2.5-0.5B with 64,000 Vocab Pruning
STUDENT_BASE_NAME = "Qwen/Qwen2.5-0.5B"
dry_log(f"[*] Initializing Student Model from: {STUDENT_BASE_NAME}...")
student_config = AutoConfig.from_pretrained(STUDENT_BASE_NAME)
student_config.vocab_size = 64000
student_model = AutoModelForCausalLM.from_config(student_config, attn_implementation="sdpa")

base_student = AutoModelForCausalLM.from_pretrained(STUDENT_BASE_NAME, torch_dtype=torch.float32, attn_implementation="sdpa")
with torch.no_grad():
    for name, param in base_student.named_parameters():
        if "embed_tokens" in name or "lm_head" in name:
            orig_weights = param.data
            subset_weights = orig_weights[new_to_old]
            dict(student_model.named_parameters())[name].data.copy_(subset_weights)
        else:
            dict(student_model.named_parameters())[name].data.copy_(param.data)

del base_student
if device.type == "cuda":
    torch.cuda.empty_cache()

student_model = student_model.to(device_student)
siglip_vision = siglip_vision.to(device_student)
unshuffle_projector = unshuffle_projector.to(device_student)
student_param_count = sum(p.numel() for p in student_model.parameters())
dry_log(f"[✓] Student Model Initialized with Pruned Weights: {student_param_count:,} parameters")
assert student_param_count == 415242112, f"Student parameter count {student_param_count} != 415,242,112 invariant!"

# Save pre-distillation weights state for comparative evaluation in Cell 10 (Auditor Item B2)
pre_distillation_state = {k: v.cpu().clone() for k, v in student_model.state_dict().items()}
dry_log("[✓] Captured Pre-Distillation Student State for Cell 10 comparative benchmark.")

# Verify Step-0 Baseline Forward Pass & Token Alignment
test_input_text = "Lisinopril 10mg tablets morning"
tokens_sub = safe_encode_64k(test_input_text)
input_ids_sub = torch.tensor([tokens_sub], device=device)

with torch.no_grad():
    student_out = student_model(input_ids_sub)
assert student_out.logits.shape[-1] == 64000, f"Expected 64k vocab logits, got {student_out.logits.shape[-1]}"
dry_log("[✓] Step-0 Student Logits Verified: Shape [1, T, 64000].")

[*] Cell 7 Start CUDA Allocated Memory: 431.27 MB
[*] Loading Multimodal Teacher: Qwen/Qwen2-VL-2B-Instruct on cuda (torch.float16)...


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (9 outputs left out)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

[✓] Student Model Initialized with Pruned Weights: 415,242,112 parameters
[✓] Captured Pre-Distillation Student State for Cell 10 comparative benchmark.
[✓] Step-0 Student Logits Verified: Shape [1, T, 64000].


In [8]:
# CELL 8: Two-Stage Multimodal Distillation (Stage 1a Projector Alignment + Stage 1b Multimodal Forward-KL)
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import hashlib
import collections
import io
import gc
from PIL import Image
from datasets import load_dataset

# Auditor Items S2 & N4: Define DATASET_LOADER_VERSION constant
DATASET_LOADER_VERSION = "3.0"

# Auditor Item R14: Log memory at start of Cell 8
if device.type == "mps" and hasattr(torch.mps, "current_allocated_memory"):
    dry_log(f"[*] Cell 8 Start MPS Allocated Memory: {torch.mps.current_allocated_memory() / (1024**2):.2f} MB")
elif device.type == "cuda":
    dry_log(f"[*] Cell 8 Start CUDA Allocated Memory: {torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")

# Auditor Item K3: Delete any existing best_student_vlm.pt at the start of cell 8
best_checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_student_vlm.pt")
if os.path.exists(best_checkpoint_path):
    try:
        os.remove(best_checkpoint_path)
        dry_log(f"[*] Cleared pre-existing checkpoint at: {best_checkpoint_path}")
    except Exception as e:
        dry_log(f"[!] Note: Could not remove pre-existing checkpoint: {e}")

# DATA ISOLATION DIRECTIVE:
# Stage 1a and 1b train on EXTERNAL datasets completely separate from BENCHMARK_SAMPLES.
# Training on benchmark samples constitutes data contamination. The 40 benchmark samples are strictly HELD-OUT.

# Auditor Item W4: Make EXTERNAL_DATA_DIR absolute
EXTERNAL_DATA_DIR = os.path.abspath("phase1_ai_sandbox/external_train_data")
os.makedirs(EXTERNAL_DATA_DIR, exist_ok=True)

# Auditor Items R6, B1, K1: Real Multimodal Dataset Loaders via datasets with streaming
def load_external_multimodal_dataset(stage="1a", train_target=2000, val_target=200):
    total_needed = train_target + val_target
    manifest_path = os.path.join(EXTERNAL_DATA_DIR, f"manifest_{stage}.json")
    
    # Auditor Items S2 & N4: Check loader version AND commit SHA in cached manifest
    if os.path.exists(manifest_path):
        try:
            with open(manifest_path, "r") as f:
                cached = json.load(f)
            cached_ver = cached.get("loader_version")
            cached_sha = cached.get("commit_sha")
            if (cached.get("loader_version") == DATASET_LOADER_VERSION and
                cached.get("commit_sha") == GIT_COMMIT_SHA and
                len(cached.get("train", [])) >= train_target and
                len(cached.get("val", [])) >= val_target):
                dry_log(f"[✓] Loaded Stage {stage} dataset from local cache (loader_version={cached_ver}, sha={cached_sha[:7]}, {len(cached['train'])} train, {len(cached['val'])} val).")
                return cached["train"][:train_target], cached["val"][:val_target]
            else:
                dry_log(f"[*] Cache invalidated for Stage {stage} (cached ver={cached_ver}, sha={str(cached_sha)[:7]} vs current ver={DATASET_LOADER_VERSION}, sha={GIT_COMMIT_SHA[:7]}). Re-streaming dataset...")
        except Exception as e:
            dry_log(f"[!] Warning reading cache {manifest_path}: {e}")

    dry_log(f"[*] Streaming real Stage {stage} multimodal dataset via datasets (target: {total_needed} pairs)...")
    
    if stage == "1a":
        dataset_candidates = [
            ("Multimodal-Fatima/COCO_captions_train", "train"),
            ("ybelkada/football-dataset", "train")
        ]
    else:
        dataset_candidates = [
            ("Multimodal-Fatima/TextVQA_train", "train"),
            ("lmms-lab-encoder/textvqa", "train")
        ]

    items_collected = []
    loaded_dataset_name = None
    loaded_dataset_split = None
    loaded_dataset_revision = "main"

    for d_name, d_split in dataset_candidates:
        try:
            dry_log(f"  --> Attempting streaming load from '{d_name}' ({d_split})...")
            stream_ds = load_dataset(d_name, split=d_split, streaming=True)
            for raw_item in stream_ds:
                if len(items_collected) >= total_needed:
                    break
                raw_img = raw_item.get("image")
                if raw_img is None:
                    continue
                if not isinstance(raw_img, Image.Image):
                    if isinstance(raw_img, dict) and "bytes" in raw_img:
                        raw_img = Image.open(io.BytesIO(raw_img["bytes"]))
                    else:
                        continue
                
                raw_img = raw_img.convert("RGB").resize((384, 384))
                # Compute SHA256 of raw image bytes for universal image grouping (Auditor Item C1)
                img_buf = io.BytesIO()
                raw_img.save(img_buf, format="JPEG")
                img_bytes = img_buf.getvalue()
                img_hash = hashlib.sha256(img_bytes).hexdigest()

                img_path = os.path.join(EXTERNAL_DATA_DIR, f"real_{stage}_{len(items_collected):05d}.jpg")
                if not os.path.exists(img_path):
                    with open(img_path, "wb") as f_img:
                        f_img.write(img_bytes)
                
                if stage == "1a":
                    cap = None
                    if "sentences_raw" in raw_item and isinstance(raw_item["sentences_raw"], (list, tuple)) and len(raw_item["sentences_raw"]) > 0:
                        cap = raw_item["sentences_raw"][0]
                    elif "caption" in raw_item and raw_item["caption"]:
                        cap = raw_item["caption"]
                    elif "sentences" in raw_item and raw_item["sentences"]:
                        sentences = raw_item["sentences"]
                        cap = sentences[0] if isinstance(sentences, (list, tuple)) and len(sentences) > 0 else str(sentences)
                    elif "text" in raw_item and raw_item["text"]:
                        cap = raw_item["text"]

                    if not cap or not str(cap).strip():
                        continue

                    cap_clean = str(cap).strip()
                    items_collected.append({"image_path": img_path, "image_hash": img_hash, "caption": cap_clean})
                    if DRY_RUN and len(items_collected) <= 2:
                        dry_log(f"  [Auditor K1 Verified] Stage 1a Sample {len(items_collected)} Caption: '{cap_clean}'")
                else:
                    q = raw_item.get("question", "What text is shown in this image?")
                    ans = raw_item.get("multiple_choice_answer")
                    if not ans and "answers" in raw_item:
                        ans_list = raw_item["answers"]
                        ans = ans_list[0] if isinstance(ans_list, list) and len(ans_list) > 0 else "text"
                    if not ans:
                        ans = "Information displayed on the image."
                    items_collected.append({"image_path": img_path, "image_hash": img_hash, "question": str(q).strip(), "answer": str(ans).strip()})

            if len(items_collected) >= total_needed:
                loaded_dataset_name = d_name
                loaded_dataset_split = d_split
                dry_log(f"  [✓] Successfully collected {len(items_collected)} real pairs from '{d_name}'.")
                break
        except Exception as e:
            dry_log(f"  [!] Candidate '{d_name}' failed: {e}")
            continue

    if len(items_collected) == 0:
        raise RuntimeError(f"Auditor Item R6 Violation: Failed to stream real images from external dataset for Stage {stage}!")

    # Auditor Item C1: Split by IMAGE HASH (not by row) to prevent data leakage across multiple questions
    groups = collections.defaultdict(list)
    for it in items_collected:
        groups[it["image_hash"]].append(it)
    unique_hashes = list(groups.keys())

    if stage == "1a":
        # plan.md §Stage 1a: Strictly 20% held-out validation split
        val_img_count = max(1, int(round(len(unique_hashes) * 0.20)))
    else:
        val_img_count = min(val_target, max(1, len(unique_hashes) // 2))

    train_hashes = unique_hashes[:-val_img_count]
    val_hashes = unique_hashes[-val_img_count:]
    train_pairs = [it for h in train_hashes for it in groups[h]]
    val_pairs = [it for h in val_hashes for it in groups[h]]

    # Auditor Item C1: Assert no image hash in both train and val
    train_set_h = set(train_hashes)
    val_set_h = set(val_hashes)
    leakage = train_set_h.intersection(val_set_h)
    assert len(leakage) == 0, f"Auditor Item C1 Violation: Data leakage! {len(leakage)} image hashes found in both train and val!"

    manifest_payload = {
        "loader_version": DATASET_LOADER_VERSION,
        "commit_sha": GIT_COMMIT_SHA,
        "stage": stage,
        "dataset_name": loaded_dataset_name or "unknown",
        "dataset_split": loaded_dataset_split or "unknown",
        "dataset_revision": loaded_dataset_revision,
        "split_train_image_hashes": list(train_hashes),
        "split_val_image_hashes": list(val_hashes),
        "train": train_pairs,
        "val": val_pairs
    }
    try:
        with open(manifest_path, "w") as f:
            json.dump(manifest_payload, f, indent=2)
        dry_log(f"[✓] Persisted Stage {stage} manifest cache (loader_version={DATASET_LOADER_VERSION}, sha={GIT_COMMIT_SHA[:7]}) to: {manifest_path}")
    except Exception as e:
        dry_log(f"[!] Warning writing manifest cache {manifest_path}: {e}")

    return train_pairs, val_pairs

# Configure dataset splits according to DRY_RUN flag
stage1a_train_target = 4 if DRY_RUN else 2000
stage1a_val_target = 2 if DRY_RUN else 200
stage1b_train_target = 4 if DRY_RUN else 1000
stage1b_val_target = 2 if DRY_RUN else 200

stage1a_train, stage1a_val = load_external_multimodal_dataset("1a", stage1a_train_target, stage1a_val_target)
stage1b_train, stage1b_val = load_external_multimodal_dataset("1b", stage1b_train_target, stage1b_val_target)

dry_log(f"[✓] Stage 1a Pairs: {len(stage1a_train)} train, {len(stage1a_val)} val")
dry_log(f"[✓] Stage 1b Pairs: {len(stage1b_train)} train, {len(stage1b_val)} val")

# -------------------------------------------------------------
# STAGE 1a: PROJECTOR ALIGNMENT (LLaVA Stage-1 Recipe)
# -------------------------------------------------------------
dry_log("="*65)
dry_log("[*] COMMENCING STAGE 1a: 2x2 SPATIAL UNSHUFFLE PROJECTOR ALIGNMENT")
dry_log("="*65)

for p in siglip_vision.parameters(): p.requires_grad = False
for p in student_model.parameters(): p.requires_grad = False
for p in unshuffle_projector.parameters(): p.requires_grad = True

# Auditor Item G2: Stage 1a Learning Rate, Warmup, Clipping & Gradient Accumulation
base_lr_1a = 1e-4
projector_optimizer = torch.optim.AdamW(unshuffle_projector.parameters(), lr=base_lr_1a, weight_decay=1e-4)
ce_criterion = nn.CrossEntropyLoss()

def evaluate_stage1a_holdout():
    unshuffle_projector.eval()
    total_loss = 0.0
    with torch.no_grad():
        for item in stage1a_val:
            raw_img = Image.open(item["image_path"]).convert("RGB").resize((384, 384))
            proc_out = siglip_processor(images=raw_img, return_tensors="pt")
            img_t = proc_out.pixel_values.to(device_student)
            if dtype_load == torch.float16: img_t = img_t.half()
            elif dtype_load == torch.bfloat16: img_t = img_t.to(torch.bfloat16)

            patches = siglip_vision(pixel_values=img_t).last_hidden_state
            v_tokens = unshuffle_projector(patches.float()).to(student_model.dtype)

            cap_ids = safe_encode_64k(item["caption"])
            cap_t = torch.tensor([cap_ids], device=device_student)
            cap_embeds = student_model.model.embed_tokens(cap_t)

            combined_embeds = torch.cat([v_tokens, cap_embeds], dim=1).to(student_model.dtype)
            logits = student_model(inputs_embeds=combined_embeds).logits

            target_ids = cap_t[:, 1:]
            pred_logits = logits[:, 144:-1, :]
            loss = ce_criterion(pred_logits.reshape(-1, 64000), target_ids.reshape(-1))
            total_loss += loss.item()
    return total_loss / max(1, len(stage1a_val))

# Auditor Item C1: Log held-out loss at >= 4 points during Stage 1a
stage1a_val_points = []
step0_val_loss = evaluate_stage1a_holdout()
stage1a_val_points.append({"step": 0, "val_loss": step0_val_loss})
dry_log(f" --> Step-0 Baseline Held-Out Stage 1a Caption Loss: {step0_val_loss:.4f}")

stage1a_steps = 2 if DRY_RUN else 200
stage1a_accum_steps = min(2 if DRY_RUN else 8, stage1a_steps)
warmup_steps_1a = max(1, int(0.10 * stage1a_steps))
eval_interval_1a = max(1, stage1a_steps // 4)
unshuffle_projector.train()
projector_optimizer.zero_grad()

for step in range(1, stage1a_steps + 1):
    lr_factor = min(1.0, step / warmup_steps_1a)
    cur_lr = base_lr_1a * lr_factor
    for pg in projector_optimizer.param_groups:
        pg["lr"] = cur_lr

    item = stage1a_train[step % len(stage1a_train)]
    raw_img = Image.open(item["image_path"]).convert("RGB").resize((384, 384))
    proc_out = siglip_processor(images=raw_img, return_tensors="pt")
    img_t = proc_out.pixel_values.to(device_student)
    if dtype_load == torch.float16: img_t = img_t.half()
    elif dtype_load == torch.bfloat16: img_t = img_t.to(torch.bfloat16)

    with torch.no_grad():
        patches = siglip_vision(pixel_values=img_t).last_hidden_state
    v_tokens = unshuffle_projector(patches.float()).to(student_model.dtype)

    cap_ids = safe_encode_64k(item["caption"])
    cap_t = torch.tensor([cap_ids], device=device_student)
    cap_embeds = student_model.model.embed_tokens(cap_t)

    combined_embeds = torch.cat([v_tokens, cap_embeds], dim=1).to(student_model.dtype)
    logits = student_model(inputs_embeds=combined_embeds).logits

    target_ids = cap_t[:, 1:]
    pred_logits = logits[:, 144:-1, :]
    loss = ce_criterion(pred_logits.reshape(-1, 64000), target_ids.reshape(-1))

    assert torch.isfinite(loss), f"Auditor Item N5 Violation: Stage 1a caption loss is not finite (got {loss.item()})!"
    loss_accum = loss / stage1a_accum_steps
    loss_accum.backward()

    if step % stage1a_accum_steps == 0 or step == stage1a_steps:
        grad_norm_1a = torch.nn.utils.clip_grad_norm_(unshuffle_projector.parameters(), 1.0)
        projector_optimizer.step()
        projector_optimizer.zero_grad()
        if (not DRY_RUN and (step % 10 == 0 or step == stage1a_steps)) or DRY_RUN:
            dry_log(f"  [Stage 1a Step {step}/{stage1a_steps}] Train Loss: {loss.item():.4f}, Grad Norm: {float(grad_norm_1a):.4f}, LR: {cur_lr:.6f}")

    # Auditor Item C1: Log held-out loss at multiple points
    if step % eval_interval_1a == 0 or step == stage1a_steps:
        mid_val = evaluate_stage1a_holdout()
        stage1a_val_points.append({"step": step, "val_loss": mid_val})
        dry_log(f"  [Stage 1a Held-Out Loss @ Step {step}/{stage1a_steps}]: {mid_val:.4f} (Baseline: {step0_val_loss:.4f})")
        unshuffle_projector.train()

final_stage1a_val = stage1a_val_points[-1]["val_loss"]
dry_log(f" --> Final Held-Out Stage 1a Caption Loss: {final_stage1a_val:.4f} (Baseline: {step0_val_loss:.4f})")
# Auditor Item K4: Replace hard assert with recorded gate result
stage1a_loss_decreased = bool(final_stage1a_val < step0_val_loss)
if stage1a_loss_decreased:
    dry_log(f"[✓] Stage 1a Projector Alignment Verified: Held-out loss strictly decreased ({step0_val_loss:.4f} -> {final_stage1a_val:.4f}).")
else:
    dry_log(f"[X] Gate Warning: Stage 1a alignment held-out loss did not strictly decrease ({step0_val_loss:.4f} -> {final_stage1a_val:.4f}). Recorded for Gate 6.")

# -------------------------------------------------------------
# STAGE 1b: MULTIMODAL BOUNDED TOP-64 FORWARD-KL DISTILLATION
# -------------------------------------------------------------
dry_log("="*65)
dry_log("[*] COMMENCING STAGE 1b: BOUNDED TOP-64 FORWARD-KL DISTILLATION")
dry_log("="*65)

for p in student_model.parameters():
    p.requires_grad = True
for p in unshuffle_projector.parameters():
    p.requires_grad = True

# Auditor Item R13: Add unshuffle_projector to optimizer
student_optimizer = torch.optim.AdamW(
    list(student_model.parameters()) + list(unshuffle_projector.parameters()),
    lr=2e-5, weight_decay=1e-4
)

def compute_bounded_top64_forward_kl(t_logits_64k, s_logits, eps=1e-5):
    """
    Auditor Item C2 & plan.md §Stage 1b exact formula:
    Omega_64 = Top-64({v in V_S | p_T(v)}).
    p~_T(v) = p_T(v) / sum_{u in Omega_64} p_T(u)  [no eps in denominator]
    p_bar_S(v) = p_S(v) / sum_{u in Omega_64} p_S(u)  [no eps in denominator]
    p~_S(v) = (1 - eps) * p_bar_S(v) + (eps / 64.0)
    KL(p~_T || p~_S) = sum_{v in Omega_64} p~_T(v) * (log(p~_T(v)) - log(p~_S(v)))  [no eps in log(p~_T)]
    Where 0 * log(0) = 0 by standard measure-theoretic limit.
    """
    p_T = F.softmax(t_logits_64k.float(), dim=-1)
    p_S = F.softmax(s_logits.float(), dim=-1)

    top64_vals, top64_indices = torch.topk(p_T, k=64, dim=-1)
    p_T_tilde = top64_vals / top64_vals.sum(dim=-1, keepdim=True)

    p_S_top64_raw = torch.gather(p_S, dim=-1, index=top64_indices)
    p_S_bar = p_S_top64_raw / p_S_top64_raw.sum(dim=-1, keepdim=True)
    p_S_tilde = (1.0 - eps) * p_S_bar + (eps / 64.0)

    term = torch.where(p_T_tilde > 0, p_T_tilde * (torch.log(p_T_tilde) - torch.log(p_S_tilde)), torch.zeros_like(p_T_tilde))
    kl = term.sum(dim=-1).mean()
    return kl

def run_kd_loss_unit_tests():
    """Auditor Item C2: CPU unit tests for Bounded Top-64 Forward-KL loss."""
    t_dummy = torch.randn(2, 5, 64000)
    s_dummy = torch.randn(2, 5, 64000)
    eps = 1e-5

    # 1. Verify both distributions sum to 1.0 (atol 1e-6)
    p_T = F.softmax(t_dummy.float(), dim=-1)
    p_S = F.softmax(s_dummy.float(), dim=-1)
    top64_vals, top64_indices = torch.topk(p_T, k=64, dim=-1)
    p_T_tilde = top64_vals / top64_vals.sum(dim=-1, keepdim=True)
    p_S_top64_raw = torch.gather(p_S, dim=-1, index=top64_indices)
    p_S_bar = p_S_top64_raw / p_S_top64_raw.sum(dim=-1, keepdim=True)
    p_S_tilde = (1.0 - eps) * p_S_bar + (eps / 64.0)

    assert torch.allclose(p_T_tilde.sum(dim=-1), torch.ones_like(p_T_tilde.sum(dim=-1)), atol=1e-6), "p_T_tilde sum != 1.0"
    assert torch.allclose(p_S_tilde.sum(dim=-1), torch.ones_like(p_S_tilde.sum(dim=-1)), atol=1e-6), "p_S_tilde sum != 1.0"

    # 2. KL >= 0 for arbitrary distributions
    kl_arb = compute_bounded_top64_forward_kl(t_dummy, s_dummy, eps=eps)
    assert kl_arb.item() >= 0.0, f"KL must be >= 0, got {kl_arb.item()}"

    # 3. KL == 0 when distributions are identical with eps=0.0
    kl_ident = compute_bounded_top64_forward_kl(t_dummy, t_dummy, eps=0.0)
    assert abs(kl_ident.item()) < 1e-6, f"KL must be 0 when identical with eps=0, got {kl_ident.item()}"

    # 4. Hand-computed 4-token example
    # p_T = [0.4, 0.3, 0.2, 0.1], p_S = [0.1, 0.2, 0.3, 0.4]
    # Hand KL = 0.4*ln(4) + 0.3*ln(1.5) + 0.2*ln(2/3) + 0.1*ln(0.25) = 0.45643482
    p_T_h = torch.tensor([[0.4, 0.3, 0.2, 0.1]])
    p_S_h = torch.tensor([[0.1, 0.2, 0.3, 0.4]])
    hand_kl_val = (p_T_h * (torch.log(p_T_h) - torch.log(p_S_h))).sum().item()

    log_t = torch.cat([torch.log(p_T_h), torch.full((1, 60), -1e9)], dim=1)
    log_s = torch.cat([torch.log(p_S_h), torch.full((1, 60), -1e9)], dim=1)
    computed_kl = compute_bounded_top64_forward_kl(log_t, log_s, eps=0.0).item()
    assert abs(computed_kl - hand_kl_val) < 1e-5, f"Hand-computed KL mismatch: expected {hand_kl_val:.6f}, got {computed_kl:.6f}"
    dry_log("[✓] Auditor Item C2 Verified: All KD loss unit tests passed.")

run_kd_loss_unit_tests()

def run_forward_distillation_step(item):
    raw_img = Image.open(item["image_path"]).convert("RGB").resize((384, 384))
    q_text = item["question"]
    ans_text = item["answer"]

    t_messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": raw_img},
                {"type": "text", "text": q_text}
            ]
        }
    ]
    t_chat_prompt = teacher_processor.apply_chat_template(t_messages, add_generation_prompt=True)
    # Auditor Item C6: Teacher on device_teacher
    t_prompt_inputs = teacher_processor(text=[t_chat_prompt], images=[raw_img], return_tensors="pt").to(device_teacher)

    student_ans_ids = safe_encode_64k(ans_text)
    L = len(student_ans_ids)
    assert L > 0, "Answer length must be > 0"
    teacher_ans_ids = [new_to_old[sid] for sid in student_ans_ids]

    t_input_ids = t_prompt_inputs["input_ids"]
    P = t_input_ids.shape[1]
    t_ans_tensor = torch.tensor([teacher_ans_ids], device=device_teacher)
    t_kwargs = {}
    for k, v in t_prompt_inputs.items():
        if k in ["input_ids", "attention_mask", "mm_token_type_ids"]:
            if k == "input_ids":
                t_kwargs[k] = torch.cat([v, t_ans_tensor], dim=1)
            elif k == "attention_mask":
                t_ones = torch.ones((1, L), device=device_teacher, dtype=v.dtype)
                t_kwargs[k] = torch.cat([v, t_ones], dim=1)
            elif k == "mm_token_type_ids":
                t_zeros = torch.zeros((1, L), device=device_teacher, dtype=v.dtype)
                t_kwargs[k] = torch.cat([v, t_zeros], dim=1)
            else:
                t_zeros = torch.zeros((1, L), device=device_teacher, dtype=v.dtype)
                t_kwargs[k] = torch.cat([v, t_zeros], dim=1)
        else:
            t_kwargs[k] = v

    t_full_input_ids = t_kwargs["input_ids"]
    assert t_full_input_ids[0, P:P+L].tolist() == teacher_ans_ids, "Teacher input_ids at answer span != teacher_ans_ids"
    decoded_t_ans = teacher_processor.tokenizer.decode(t_full_input_ids[0, P:P+L]).strip()
    assert decoded_t_ans == ans_text.strip(), f"Decoded teacher answer '{decoded_t_ans}' != expected '{ans_text.strip()}'"

    assert teacher_model is not None, "Auditor Item A8 Violation: Teacher model cannot be None during distillation!"
    with torch.no_grad():
        t_out = teacher_model(**t_kwargs)

    t_logits_ans = t_out.logits[:, P-1:P-1+L, :]
    assert torch.isfinite(t_logits_ans).all(), "Auditor Item KB3 Violation: Teacher logits contain inf or nan (fp16 overflow)!"
    # Auditor Item C6: Move only teacher answer logits to device_student before KL
    t_logits_64k = t_logits_ans[:, :, new_to_old].to(device_student)

    # Student on device_student
    proc_out = siglip_processor(images=raw_img, return_tensors="pt")
    img_t = proc_out.pixel_values.to(device_student)
    if dtype_load == torch.float16: img_t = img_t.half()
    elif dtype_load == torch.bfloat16: img_t = img_t.to(torch.bfloat16)

    with torch.no_grad():
        patches = siglip_vision(pixel_values=img_t).last_hidden_state
    v_tokens = unshuffle_projector(patches.float()).to(student_model.dtype)

    q_ids = safe_encode_64k(q_text)
    Q = len(q_ids)
    text_ids = torch.tensor([q_ids + student_ans_ids], device=device_student)
    text_embeds = student_model.model.embed_tokens(text_ids)

    combined = torch.cat([v_tokens, text_embeds], dim=1).to(student_model.dtype)
    s_logits_full = student_model(inputs_embeds=combined).logits

    start_ans_idx = 144 + Q - 1
    s_logits_ans = s_logits_full[:, start_ans_idx:start_ans_idx+L, :]

    assert s_logits_ans.shape[1] == t_logits_64k.shape[1] == L, f"Answer length mismatch: student={s_logits_ans.shape[1]}, teacher={t_logits_64k.shape[1]}"

    kl = compute_bounded_top64_forward_kl(t_logits_64k, s_logits_ans)
    assert torch.isfinite(kl), "Auditor Item KB3 Violation: Forward-KL loss is not finite (inf or nan)!"
    return kl

# Auditor Item C3: Precompute teacher top-64 targets & SigLIP patches for validation set ONCE
dry_log(f"[*] Precomputing teacher top-64 targets and SigLIP patches for {len(stage1b_val)} validation items (Auditor Item C3)...")
cached_stage1b_val = []
for v_idx, item in enumerate(stage1b_val):
    raw_img = Image.open(item["image_path"]).convert("RGB").resize((384, 384))
    q_text = item["question"]
    ans_text = item["answer"]

    t_messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": raw_img},
                {"type": "text", "text": q_text}
            ]
        }
    ]
    t_chat_prompt = teacher_processor.apply_chat_template(t_messages, add_generation_prompt=True)
    t_prompt_inputs = teacher_processor(text=[t_chat_prompt], images=[raw_img], return_tensors="pt").to(device_teacher)

    student_ans_ids = safe_encode_64k(ans_text)
    L = len(student_ans_ids)
    teacher_ans_ids = [new_to_old[sid] for sid in student_ans_ids]

    t_input_ids = t_prompt_inputs["input_ids"]
    P = t_input_ids.shape[1]
    t_ans_tensor = torch.tensor([teacher_ans_ids], device=device_teacher)
    t_kwargs = {}
    for k, v in t_prompt_inputs.items():
        if k in ["input_ids", "attention_mask", "mm_token_type_ids"]:
            if k == "input_ids":
                t_kwargs[k] = torch.cat([v, t_ans_tensor], dim=1)
            elif k == "attention_mask":
                t_ones = torch.ones((1, L), device=device_teacher, dtype=v.dtype)
                t_kwargs[k] = torch.cat([v, t_ones], dim=1)
            elif k == "mm_token_type_ids":
                t_zeros = torch.zeros((1, L), device=device_teacher, dtype=v.dtype)
                t_kwargs[k] = torch.cat([v, t_zeros], dim=1)
            else:
                t_zeros = torch.zeros((1, L), device=device_teacher, dtype=v.dtype)
                t_kwargs[k] = torch.cat([v, t_zeros], dim=1)
        else:
            t_kwargs[k] = v

    with torch.no_grad():
        t_out = teacher_model(**t_kwargs)
    t_logits_ans = t_out.logits[:, P-1:P-1+L, :]
    assert torch.isfinite(t_logits_ans).all(), "Teacher logits contain inf or nan!"
    t_logits_64k = t_logits_ans[:, :, new_to_old].to(device_student)

    # Renormalized teacher probabilities over top-64
    p_T = F.softmax(t_logits_64k.float(), dim=-1)
    top64_vals, top64_indices = torch.topk(p_T, k=64, dim=-1)
    p_T_tilde = top64_vals / top64_vals.sum(dim=-1, keepdim=True)

    # Precompute SigLIP patches
    proc_out = siglip_processor(images=raw_img, return_tensors="pt")
    img_t = proc_out.pixel_values.to(device_student)
    if dtype_load == torch.float16: img_t = img_t.half()
    elif dtype_load == torch.bfloat16: img_t = img_t.to(torch.bfloat16)
    with torch.no_grad():
        patches = siglip_vision(pixel_values=img_t).last_hidden_state.detach().cpu()

    q_ids = safe_encode_64k(q_text)
    cached_stage1b_val.append({
        "patches": patches,
        "top64_indices": top64_indices.detach().to(device_student),
        "p_T_tilde": p_T_tilde.detach().to(device_student),
        "q_ids": q_ids,
        "student_ans_ids": student_ans_ids,
        "L": L
    })
dry_log(f"[✓] Auditor Item C3 Verified: Cached {len(cached_stage1b_val)} validation targets; teacher forwards eliminated during validation.")

def evaluate_kd_holdout(eps=1e-5):
    """Auditor Item C3: Efficient validation without redundant teacher forward passes."""
    student_model.eval()
    unshuffle_projector.eval()
    total_kl = 0.0
    with torch.no_grad():
        for cached in cached_stage1b_val:
            patches = cached["patches"].to(device_student)
            v_tokens = unshuffle_projector(patches.float()).to(student_model.dtype)

            q_ids = cached["q_ids"]
            student_ans_ids = cached["student_ans_ids"]
            L = cached["L"]
            Q = len(q_ids)
            text_ids = torch.tensor([q_ids + student_ans_ids], device=device_student)
            text_embeds = student_model.model.embed_tokens(text_ids)

            combined = torch.cat([v_tokens, text_embeds], dim=1).to(student_model.dtype)
            s_logits_full = student_model(inputs_embeds=combined).logits

            start_ans_idx = 144 + Q - 1
            s_logits_ans = s_logits_full[:, start_ans_idx:start_ans_idx+L, :]

            p_S = F.softmax(s_logits_ans.float(), dim=-1)
            p_S_top64_raw = torch.gather(p_S, dim=-1, index=cached["top64_indices"])
            p_S_bar = p_S_top64_raw / p_S_top64_raw.sum(dim=-1, keepdim=True)
            p_S_tilde = (1.0 - eps) * p_S_bar + (eps / 64.0)

            p_T_tilde = cached["p_T_tilde"]
            term = torch.where(p_T_tilde > 0, p_T_tilde * (torch.log(p_T_tilde) - torch.log(p_S_tilde)), torch.zeros_like(p_T_tilde))
            kl = term.sum(dim=-1).mean()
            assert torch.isfinite(kl), "Holdout KL loss is not finite!"
            total_kl += kl.item()
    return total_kl / max(1, len(cached_stage1b_val))

step0_kd_val = evaluate_kd_holdout()
dry_log(f" --> Step-0 Held-Out Multimodal KD Loss: {step0_kd_val:.4f}")

best_val_loss = step0_kd_val

stage1b_steps = 2 if DRY_RUN else 100
dry_log(f"[*] Distilling with gradient accumulation over {stage1b_steps} steps...")
student_model.train()
unshuffle_projector.train()
student_optimizer.zero_grad()
accum_steps = min(2 if DRY_RUN else 8, stage1b_steps)

for step in range(1, stage1b_steps + 1):
    item = stage1b_train[step % len(stage1b_train)]
    loss = run_forward_distillation_step(item)
    loss_accum = loss / accum_steps
    # Auditor Item KB3: Assert finite loss before backward pass
    assert torch.isfinite(loss_accum), "Auditor Item KB3 Violation: Forward-KL loss is not finite before backward!"
    loss_accum.backward()

    # Watchlist W3: Log MPS memory during Stage 1b
    if step == 1 or step == stage1b_steps:
        if device.type == "mps" and hasattr(torch.mps, "current_allocated_memory"):
            dry_log(f"  [W3 MPS Peak] Stage 1b Step {step} MPS Allocated: {torch.mps.current_allocated_memory() / (1024**2):.2f} MB")

    if step % accum_steps == 0 or step == stage1b_steps:
        # Auditor Item G2: Gradient clipping max norm 1.0 for Stage 1b
        active_params_1b = [p for p in list(student_model.parameters()) + list(unshuffle_projector.parameters()) if p.requires_grad]
        grad_norm_1b = torch.nn.utils.clip_grad_norm_(active_params_1b, 1.0)
        student_optimizer.step()
        student_optimizer.zero_grad()
        
        # Auditor Item R5: Evaluate on val set after each optimizer step; update and save ONLY on strict improvement
        cur_val_loss = evaluate_kd_holdout()
        student_model.train()
        unshuffle_projector.train()
        if cur_val_loss < best_val_loss:
            best_val_loss = cur_val_loss
            torch.save({
                "student_state_dict": student_model.state_dict(),
                "projector_state_dict": unshuffle_projector.state_dict(),
                "best_val_loss": best_val_loss,
                "step": step
            }, best_checkpoint_path)
            dry_log(f"  [✓] Step {step}: Held-out validation loss strictly improved to {best_val_loss:.4f} -> Checkpoint saved.")

    if step % 2 == 0 or step == stage1b_steps:
        dry_log(f"  [Stage 1b Step {step}/{stage1b_steps}] Forward-KL Loss: {loss.item():.4f}, Grad Norm: {float(grad_norm_1b if 'grad_norm_1b' in locals() else 0.0):.4f}")

final_kd_val = evaluate_kd_holdout()
dry_log(f" --> Final Held-Out Multimodal KD Loss: {final_kd_val:.4f} (Best: {best_val_loss:.4f}, Step-0 Baseline: {step0_kd_val:.4f})")
dry_log(f"[✓] Checkpoint state persisted at: {best_checkpoint_path}")

# Auditor Item C6: Device memory and map telemetry
if torch.cuda.is_available():
    dry_log(f"[*] Teacher hf_device_map: {getattr(teacher_model, 'hf_device_map', None)}")
    for d_idx in range(torch.cuda.device_count()):
        dry_log(f"[*] CUDA Device {d_idx} ({torch.cuda.get_device_name(d_idx)}) Max Memory Allocated: {torch.cuda.max_memory_allocated(d_idx) / (1024**2):.2f} MB")

[*] Cell 8 Start CUDA Allocated Memory: 2017.98 MB
[*] Streaming real Stage 1a multimodal dataset via datasets (target: 2200 pairs)...
  --> Attempting streaming load from 'Multimodal-Fatima/COCO_captions_train' (train)...


README.md:   0%|          | 0.00/758 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

  [✓] Successfully collected 2200 real pairs from 'Multimodal-Fatima/COCO_captions_train'.
[✓] Persisted Stage 1a manifest cache (loader_version=3.0, sha=d5f80bf) to: /kaggle/working/phase1_ai_sandbox/external_train_data/manifest_1a.json
[*] Streaming real Stage 1b multimodal dataset via datasets (target: 1200 pairs)...
  --> Attempting streaming load from 'Multimodal-Fatima/TextVQA_train' (train)...


README.md:   0%|          | 0.00/891 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

  [✓] Successfully collected 1200 real pairs from 'Multimodal-Fatima/TextVQA_train'.
[✓] Persisted Stage 1b manifest cache (loader_version=3.0, sha=d5f80bf) to: /kaggle/working/phase1_ai_sandbox/external_train_data/manifest_1b.json
[✓] Stage 1a Pairs: 1760 train, 440 val
[✓] Stage 1b Pairs: 880 train, 320 val
[*] COMMENCING STAGE 1a: 2x2 SPATIAL UNSHUFFLE PROJECTOR ALIGNMENT
 --> Step-0 Baseline Held-Out Stage 1a Caption Loss: 6.3472
  [Stage 1a Step 40/200] Train Loss: 4.7863, Grad Norm: 22.8850, LR: 0.000100
  [Stage 1a Held-Out Loss @ Step 50/200]: 5.5480 (Baseline: 6.3472)
  [Stage 1a Step 80/200] Train Loss: 3.6497, Grad Norm: 221.8941, LR: 0.000100
  [Stage 1a Held-Out Loss @ Step 100/200]: 5.2052 (Baseline: 6.3472)
  [Stage 1a Step 120/200] Train Loss: 4.5434, Grad Norm: 26.6105, LR: 0.000100
  [Stage 1a Held-Out Loss @ Step 150/200]: 4.9452 (Baseline: 6.3472)
  [Stage 1a Step 160/200] Train Loss: 2.9318, Grad Norm: 11.0234, LR: 0.000100
  [Stage 1a Step 200/200] Train Loss: 2.5

In [9]:
# CELL 9: Checkpoint Serialization & Multimodal Forward Pass Verification
import os
import torch

# Auditor Item K3 & C4: Load both student_state_dict AND projector_state_dict from the same checkpoint FIRST
if os.path.exists(best_checkpoint_path):
    ckpt = torch.load(best_checkpoint_path, map_location=device_student)
    if isinstance(ckpt, dict):
        if "student_state_dict" in ckpt:
            student_model.load_state_dict(ckpt["student_state_dict"])
            dry_log(f"[✓] Loaded student_state_dict from best validation checkpoint.")
        if "projector_state_dict" in ckpt:
            unshuffle_projector.load_state_dict(ckpt["projector_state_dict"])
            dry_log(f"[✓] Loaded projector_state_dict from best validation checkpoint.")
    else:
        student_model.load_state_dict(ckpt)
    dry_log(f"[✓] Loaded best validation checkpoint from: {best_checkpoint_path}")

# Auditor Item C4: Save spatial_unshuffle_downsampler.pt AFTER loading the best checkpoint
downsampler_path = os.path.join(CHECKPOINT_DIR, "spatial_unshuffle_downsampler.pt")
torch.save(unshuffle_projector.state_dict(), downsampler_path)
dry_log(f"[✓] Spatial Unshuffle Downsampler saved to: {downsampler_path} (from best checkpoint)")

student_model.eval()
unshuffle_projector.eval()

# Auditor Item C4: Deployment-loader check matching live_wearable_emulator.py
dry_log("[*] Verifying deployment loader parity with live_wearable_emulator.py...")
reloaded_projector = SpatialUnshuffleDownsampler(in_features=768, out_features=896, grid_size=24).to(device_student)
reloaded_downsampler_state = torch.load(downsampler_path, map_location=device_student)
# Strict loading check - no silent strict=False
reloaded_projector.load_state_dict(reloaded_downsampler_state, strict=True)
reloaded_projector.eval()

# End-to-end forward verification
test_img = torch.randn(1, 3, 384, 384, device=device_student, dtype=dtype_load)
with torch.no_grad():
    p = siglip_vision(pixel_values=test_img).last_hidden_state
    v = unshuffle_projector(p.float())
    v_reloaded = reloaded_projector(p.float())
    proj_diff = (v - v_reloaded).abs().max().item()
    dry_log(f"[✓] Auditor Item C4 Verified: Projector deployment loader check (max abs diff: {proj_diff:.2e})")
    assert proj_diff < 1e-6, f"Auditor Item C4 Violation: Deployment loader mismatch! (max abs diff {proj_diff} >= 1e-6)"

    q_ids = torch.tensor([[safe_encode_64k("Read label")[0]]], device=device_student)
    q_emb = student_model.model.embed_tokens(q_ids)
    c = torch.cat([v, q_emb], dim=1)
    out = student_model(inputs_embeds=c)

assert out.logits.shape == (1, 145, 64000), f"Output shape mismatch: {out.logits.shape}"
dry_log(f"[✓] End-to-End Multimodal Forward Pass Verified: Shape {list(out.logits.shape)}")

[✓] Loaded student_state_dict from best validation checkpoint.
[✓] Loaded projector_state_dict from best validation checkpoint.
[✓] Loaded best validation checkpoint from: /kaggle/working/FRIDAY_weights/best_student_vlm.pt
[✓] Spatial Unshuffle Downsampler saved to: /kaggle/working/FRIDAY_weights/spatial_unshuffle_downsampler.pt (from best checkpoint)
[*] Verifying deployment loader parity with live_wearable_emulator.py...
[✓] Auditor Item C4 Verified: Projector deployment loader check (max abs diff: 0.00e+00)
[✓] End-to-End Multimodal Forward Pass Verified: Shape [1, 145, 64000]


In [10]:
# CELL 10: End-to-End Multimodal Inference & Comparative Benchmark
from PIL import Image
import numpy as np
import torch
import gc
import re
from transformers import AutoModelForCausalLM, Qwen2ForCausalLM
from qwen_vl_utils import process_vision_info

# Auditor Items R14 & W3: Log memory and free optimizer & cache before building pre_student_model
if device.type == "mps" and hasattr(torch.mps, "current_allocated_memory"):
    dry_log(f"[*] Cell 10 Start MPS Allocated Memory: {torch.mps.current_allocated_memory() / (1024*1024):.2f} MB")
elif device.type == "cuda":
    dry_log(f"[*] Cell 10 Start CUDA Allocated Memory: {torch.cuda.memory_allocated() / (1024*1024):.2f} MB")

if "student_optimizer" in globals():
    del student_optimizer
if "projector_optimizer" in globals():
    del projector_optimizer
gc.collect()
if device.type == "mps" and hasattr(torch.mps, "empty_cache"):
    torch.mps.empty_cache()
elif device.type == "cuda":
    torch.cuda.empty_cache()

# Re-instantiate untrained pre-distillation student baseline (543234b reference)
pre_student_model = AutoModelForCausalLM.from_config(student_config, attn_implementation="sdpa").to(device_student)
pre_student_model.load_state_dict({k: v.to(device_student) for k, v in pre_distillation_state.items()})
pre_student_model.eval()

student_model.eval()
unshuffle_projector.eval()

# Auditor Item E1: Ensure correct EOS token ID mapped to 64k vocabulary
assert base_tok.eos_token_id in old_to_new, "Auditor Item E1 Violation: EOS token ID missing from 64k vocabulary subset!"
STUDENT_EOS_ID = old_to_new[base_tok.eos_token_id]

def truncate_at_eos(text: str) -> str:
    """Auditor Item E1: Truncates decoded output at the first EOS or turn token."""
    for marker in ["<|endoftext|>", "<|im_end|>", "<|endofprompt|>"]:
        if marker in text:
            text = text.split(marker)[0]
    return text.strip()

def generate_literal_answer(model, prompt_text, max_new_tokens=48):
    input_ids = torch.tensor([safe_encode_64k(prompt_text)], device=device_student)
    # Auditor Item C5: Pass explicit attention_mask and remove temperature/top_p/top_k when do_sample=False
    attn_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            max_new_tokens=max_new_tokens,
            eos_token_id=STUDENT_EOS_ID,
            pad_token_id=STUDENT_EOS_ID,
            do_sample=False
        )
    gen_tokens = out[0][input_ids.shape[1]:].tolist()
    return truncate_at_eos(decode_64k(gen_tokens).strip())

def generate_vision_answer(model, image_path, question_text, max_new_tokens=48):
    """End-to-end multimodal student inference with correct 64k EOS token ID."""
    raw_img = Image.open(image_path).convert("RGB").resize((384, 384))
    proc_out = siglip_processor(images=raw_img, return_tensors="pt")
    img_t = proc_out.pixel_values.to(device_student)
    if dtype_load == torch.float16: img_t = img_t.half()
    elif dtype_load == torch.bfloat16: img_t = img_t.to(torch.bfloat16)
    with torch.no_grad():
        patches = siglip_vision(pixel_values=img_t).last_hidden_state
        v_tokens = unshuffle_projector(patches.float()).to(model.dtype)
        q_ids = safe_encode_64k(question_text)
        q_t = torch.tensor([q_ids], device=device_student)
        q_embeds = model.model.embed_tokens(q_t)
        inputs_embeds = torch.cat([v_tokens, q_embeds], dim=1).to(model.dtype)
        curr_embeds = inputs_embeds
        gen_tokens = []
        for _ in range(max_new_tokens):
            logits = model(inputs_embeds=curr_embeds).logits[:, -1, :]
            next_id = int(torch.argmax(logits, dim=-1).item())
            if next_id == STUDENT_EOS_ID:
                break
            gen_tokens.append(next_id)
            next_tensor = torch.tensor([[next_id]], device=device_student)
            next_token_emb = model.model.embed_tokens(next_tensor)
            curr_embeds = torch.cat([curr_embeds, next_token_emb], dim=1)
    return truncate_at_eos(decode_64k(gen_tokens).strip())

def generate_teacher_answer(image_path, prompt_text):
    """Qwen2-VL-2B-Instruct teacher model ground-truth inference."""
    raw_img = Image.open(image_path).convert("RGB").resize((384, 384))
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": raw_img},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    txt = teacher_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img_inputs, _ = process_vision_info(messages)
    teacher_inputs = teacher_processor(text=[txt], images=img_inputs, padding=True, return_tensors="pt").to(device_teacher)
    with torch.no_grad():
        out = teacher_model.generate(
            **teacher_inputs,
            max_new_tokens=48,
            do_sample=False
        )
    gen = out[0][teacher_inputs.input_ids.shape[1]:].tolist()
    return truncate_at_eos(teacher_processor.decode(gen, skip_special_tokens=True))

# Auditor Items A2 & A3: Run unit tests for field-level scoring
run_score_response_unit_tests()
dry_log("[✓] All Auditor Items A2 & A3 score_response unit tests passed successfully!")

lit_counts = {"correct": 0, "safe_refusal": 0, "wrong_spoken": 0, "incomplete": 0, "no_answer": 0}
vis_counts = {"correct": 0, "safe_refusal": 0, "wrong_spoken": 0, "incomplete": 0, "no_answer": 0}
pre_counts = {"correct": 0, "safe_refusal": 0, "wrong_spoken": 0, "incomplete": 0, "no_answer": 0}
tea_counts = {"correct": 0, "safe_refusal": 0, "wrong_spoken": 0, "incomplete": 0, "no_answer": 0}

dry_log("\n[*] EVALUATING MODELS ON BENCHMARK DATASET (Auditor Items A2, A3, A6, B2, B3):")
eval_subset = BENCHMARK_SAMPLES[:4] if DRY_RUN else BENCHMARK_SAMPLES

for idx, item in enumerate(eval_subset):
    q_prompt = item.get("prompt", "What is shown in this image?")
    noisy_ocr = benchmark_ocr_results[idx]["full_ocr"]
    target_key = item["key_field_value"]
    gt_full_text = item.get("ground_truth_full_text", "")
    ocr_lines = benchmark_ocr_results[idx].get("ocr_lines", [])

    # Product Path: Extractive Literal Reader (Auditor Item B3)
    lit_res = read_literal(ocr_lines, item["category"], q_prompt)
    lit_ans = lit_res.get("speech", REFUSAL_TEXT)
    fb = (lit_ans == REFUSAL_TEXT)

    # Comparison Row: Student Free-Gen (NOT the product path)
    lit_input_prompt = f"{q_prompt} Context: {noisy_ocr}\nAnswer:"
    comp_free_gen_ans = generate_literal_answer(student_model, lit_input_prompt)
    pre_ans = generate_literal_answer(pre_student_model, lit_input_prompt)

    vis_input_prompt = f"{q_prompt}\nAnswer:"
    vis_ans = generate_vision_answer(student_model, item["image_path"], vis_input_prompt)
    tea_ans = generate_teacher_answer(item["image_path"], q_prompt)

    lit_verdict = score_response(lit_ans, item, gt_full_text, is_fallback=fb)
    comp_verdict = score_response(comp_free_gen_ans, item, gt_full_text, is_fallback=False)
    vis_verdict = score_response(vis_ans, item, gt_full_text, is_fallback=False)
    pre_verdict = score_response(pre_ans, item, gt_full_text, is_fallback=fb)
    tea_verdict = score_response(tea_ans, item, gt_full_text, is_fallback=False)

    lit_counts[lit_verdict] += 1
    vis_counts[vis_verdict] += 1
    pre_counts[pre_verdict] += 1
    tea_counts[tea_verdict] += 1

    if idx < 4:
        dry_log(f"\n--- Sample [{item['id']}]: {item['title']} ---")
        dry_log(f"  Target Key              : '{item['key_field_value']}' (Safety: '{item.get('expected_safety_entity', '')}')")
        dry_log(f"  Literal Path (Product)  : '{lit_ans}' -> {lit_verdict}")
        dry_log(f"  Student Free-Gen (Comp) : '{comp_free_gen_ans}' -> {comp_verdict} (student free-gen, NOT the product path)")
        dry_log(f"  Vision Path Ans         : '{vis_ans}' -> {vis_verdict}")
        dry_log(f"  Pre-Distill Ans         : '{pre_ans}' -> {pre_verdict}")
        dry_log(f"  Teacher Model Ans       : '{tea_ans}' -> {tea_verdict}")

dry_log("\n" + "="*65)
dry_log("END-TO-END PIPELINE MODEL SCORING & SAFETY AUDIT (A2, A3, B2, B3)")
dry_log("="*65)
dry_log(f"  Literal Path (Product)  : Correct = {lit_counts['correct']}, Safe-Refusal = {lit_counts['safe_refusal']}, Wrong-Spoken = {lit_counts['wrong_spoken']}, Incomplete = {lit_counts['incomplete']}, No-Answer = {lit_counts['no_answer']}")
dry_log(f"  Student Free-Gen (Comp) : (student free-gen, NOT the product path)")
dry_log(f"  Vision Path             : Correct = {vis_counts['correct']}, Safe-Refusal = {vis_counts['safe_refusal']}, Wrong-Spoken = {vis_counts['wrong_spoken']}, Incomplete = {vis_counts['incomplete']}, No-Answer = {vis_counts['no_answer']}")
dry_log(f"  Pre-Distill             : Correct = {pre_counts['correct']}, Safe-Refusal = {pre_counts['safe_refusal']}, Wrong-Spoken = {pre_counts['wrong_spoken']}, Incomplete = {pre_counts['incomplete']}, No-Answer = {pre_counts['no_answer']}")
dry_log(f"  Teacher Model           : Correct = {tea_counts['correct']}, Safe-Refusal = {tea_counts['safe_refusal']}, Wrong-Spoken = {tea_counts['wrong_spoken']}, Incomplete = {tea_counts['incomplete']}, No-Answer = {tea_counts['no_answer']}")

lit_acc = (lit_counts["correct"] / max(1, len(eval_subset))) * 100.0
saf_acc = ((lit_counts["correct"] + lit_counts["safe_refusal"]) / max(1, len(eval_subset))) * 100.0
dry_log(f"  Literal Path Key Accuracy: {lit_acc:.1f}%")
dry_log("="*65)

# Auditor Item C6: Device memory telemetry at end of Cell 10
if torch.cuda.is_available():
    dry_log(f"[*] Teacher hf_device_map: {getattr(teacher_model, 'hf_device_map', None)}")
    for d_idx in range(torch.cuda.device_count()):
        dry_log(f"[*] CUDA Device {d_idx} ({torch.cuda.get_device_name(d_idx)}) Max Memory Allocated: {torch.cuda.max_memory_allocated(d_idx) / (1024**2):.2f} MB")

[*] Cell 10 Start CUDA Allocated Memory: 6934.53 MB
[✓] All Auditor Items A2 & A3 score_response unit tests passed successfully!

[*] EVALUATING MODELS ON BENCHMARK DATASET (Auditor Items A2, A3, A6, B2, B3):


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



--- Sample [med_01]: Amoxicillin 500mg Capsule Bottle ---
  Target Key              : 'Amoxicillin 500mg' (Safety: '500mg')
  Literal Path (Product)  : 'AMOXICILLIN 500 MG CAPSULE. TAKE 1 CAPSULE BY MOUTH TWICE DAILY FOR 10 DAYS.' -> correct
  Student Free-Gen (Comp) : 'The medication prescribed for ELEANOR RIGBY is Metoprolol (Amoxicillin) Capsules. The prescribed dose is 500 mg per capsule. The medication is taken twice daily for 10 days.' -> wrong_spoken (student free-gen, NOT the product path)
  Vision Path Ans         : 'The prescribed dose is 1 tablet twice daily.
The prescribed dose is 1 tablet twice daily.
The prescribed dose is 1 tablet twice daily.
The prescribed dose is 1 tablet twice daily.
The prescribed dose is 1 tablet twice' -> wrong_spoken
  Pre-Distill Ans         : 'Metoprolol (500 mg) is a medication prescribed for the treatment of hypertension. The recommended dose is 1 capsule taken twice daily for 10 days. It is important to follow the instructions provided by t

In [11]:
# CELL 11: Real Packed W4A16 Quantization & Binary Serialization (Auditor Item B3)
import os

PACKED_WEIGHTS_PATH = os.path.join(CHECKPOINT_DIR, "friday_student_w4a16_packed.pt")
dry_log(f"[*] Serializing True All-INT4 Packed Binary (including tied embed_tokens/lm_head) to: {PACKED_WEIGHTS_PATH}...")

packed_state = {}
total_packed_bytes = 0

with torch.no_grad():
    for name, param in student_model.named_parameters():
        if "weight" in name and ("layers" in name or "lm_head" in name or "embed_tokens" in name):
            w = param.data.float()
            w_min = w.min(dim=-1, keepdim=True)[0]
            w_max = w.max(dim=-1, keepdim=True)[0]
            scale = (w_max - w_min) / 15.0
            scale = torch.clamp(scale, min=1e-8)
            q = torch.clamp(torch.round((w - w_min) / scale), 0, 15).to(torch.uint8)

            q_flat = q.reshape(-1)
            if q_flat.numel() % 2 != 0:
                q_flat = torch.cat([q_flat, torch.zeros(1, dtype=torch.uint8, device=device)])
            packed = (q_flat[0::2] << 4) | (q_flat[1::2] & 0x0F)

            packed_state[f"{name}_packed"] = packed.cpu()
            packed_state[f"{name}_scale"] = scale.cpu().half()
            packed_state[f"{name}_min"] = w_min.cpu().half()
            total_packed_bytes += packed.numel()
        else:
            packed_state[name] = param.data.cpu().half()

torch.save(packed_state, PACKED_WEIGHTS_PATH)
artifact_bytes = os.path.getsize(PACKED_WEIGHTS_PATH)
artifact_mib = artifact_bytes / (1024 * 1024)
artifact_mb = artifact_bytes / 1e6

dry_log(f"[✓] Physical Artifact Written to Disk: {PACKED_WEIGHTS_PATH}")
dry_log(f"  Binary File Size: {artifact_mib:.2f} MiB ({artifact_mb:.2f} MB)")
dry_log(f"  Target Theoretical Footprint: ~208.63 MB (Contract Limit: <= 233.00 MB)")

assert artifact_mb <= 233.0, f"Auditor Gate B3 Failure: Packed artifact size {artifact_mb:.2f} MB exceeds 233.0 MB!"
dry_log("[✓] Artifact size verified strictly below 233.0 MB limit.")

[*] Serializing True All-INT4 Packed Binary (including tied embed_tokens/lm_head) to: /kaggle/working/FRIDAY_weights/friday_student_w4a16_packed.pt...
[✓] Physical Artifact Written to Disk: /kaggle/working/FRIDAY_weights/friday_student_w4a16_packed.pt
  Binary File Size: 199.68 MiB (209.38 MB)
  Target Theoretical Footprint: ~208.63 MB (Contract Limit: <= 233.00 MB)
[✓] Artifact size verified strictly below 233.0 MB limit.


In [12]:
# CELL 12: Milestone 1.5 Final Unified Gate Execution & Report Export
import json
import time
import os

dry_log("="*65)
dry_log("PROJECT FRIDAY: MILESTONE 1.5 UNIFIED FINANCIAL & TECHNICAL GATE")
dry_log("="*65)

# Gate 1: Optical Resolution & Foveation (Analytical Physics Contract, Auditor Item R7)
reading_dist_cm = 35.0
char_height_mm = 1.8
imx415_pixel_pitch_um = 1.45
focal_length_mm = 3.6
px_per_char = (char_height_mm * focal_length_mm) / (reading_dist_cm * 10.0 * imx415_pixel_pitch_um * 1e-3)
g1_pass = (px_per_char >= 10.0)

# Gate 2: 2x2 Spatial Unshuffle Token Invariant & Host Latency Contract (plan.md §5.1, §5.2)
test_patches_lat = torch.randn(1, 576, 768, device=device_student, dtype=torch.float32)
for _ in range(5):
    _ = unshuffle_projector(test_patches_lat)
device_sync()
t0_lat = time.perf_counter()
num_lat_iters = 50
for _ in range(num_lat_iters):
    _ = unshuffle_projector(test_patches_lat)
device_sync()
downsampler_latency_ms = ((time.perf_counter() - t0_lat) / num_lat_iters) * 1000.0
g2_latency_pass = bool(downsampler_latency_ms < 10.0)
g2_pass = bool((projected_tokens.shape[1] == 144) and (projected_tokens.shape[2] == 896) and g2_latency_pass)

# Gate 3: Memory Bandwidth Contract (Analytic Contract from 415M Student Parameters, plan.md §5.2)
# Auditor Item C8: Computed purely in raw bytes with decimal MB/GB conversion
param_count_int4 = 415242112
weights_bytes = param_count_int4 * 0.5  # 207,621,056 bytes
scales_bytes = 16000000  # 16.0 MB decimal
kv_cache_bytes = 2 * 24 * 2 * 64 * 1 * 1024  # 6,291,456 bytes
total_bytes_per_tok = weights_bytes + scales_bytes + kv_cache_bytes  # 229,912,512 bytes
total_mb_per_tok = total_bytes_per_tok / 1e6  # 229.91 MB
operating_bw = (total_bytes_per_tok * 25.0) / 1e9  # 5.75 GB/s @ 25 tok/s
bus_ceiling = 14.00  # Safe Operational Ceiling (Firefly CAM-3576Q38 32-bit LPDDR5 @ 5500 MT/s)
g3_pass = bool(operating_bw < bus_ceiling)

# Gate 4: Kokoro Voice Quality
g4_status = "NOT TESTED (Kokoro runtime not loaded in this session)"
g4_pass = False

# Gate 5: Downstream 40-Sample Benchmark Accuracy & Safety Protocol (Auditor Items A6 & B2)
raw_cer_val = float(np.mean(all_raw_cers) * 100.0)
raw_wer_val = float(np.mean(all_raw_wers) * 100.0)
g5_wrong_spoken_pass = (lit_counts["wrong_spoken"] == 0)
g5_pass = (lit_acc >= 95.0) and (saf_acc >= 95.0) and (raw_cer_val <= 5.0) and (raw_wer_val <= 30.0) and g5_wrong_spoken_pass

# Gate 6: Multimodal Distillation Convergence (Auditor Items A8, R5, K4)
# In DRY_RUN, Stage 1a strictly recorded from stage1a_loss_decreased; Stage 1b loss non-negative
if DRY_RUN:
    g6_pass = stage1a_loss_decreased and (best_val_loss <= step0_kd_val)
else:
    g6_pass = stage1a_loss_decreased and (best_val_loss <= step0_kd_val * 0.95)

overall_gate_pass = bool(g1_pass and g2_pass and g3_pass and g4_pass and g5_pass and g6_pass)

dry_log(f"  [GATE 1] Optical Resolution (>=10 px/char)      : {'PASS [✓]' if g1_pass else 'FAIL [X]'} ({px_per_char:.1f} px/char, Analytic)")
dry_log(f"  [GATE 2] Spatial Unshuffle (strictly 144 tokens) : {'PASS [✓]' if g2_pass else 'FAIL [X]'}")
dry_log(f"  [GATE 3] Memory Bandwidth (<14.00 GB/s)          : {'PASS [✓]' if g3_pass else 'FAIL [X]'} ({operating_bw:.2f} GB/s @ 25 tok/s, Analytic)")
dry_log(f"  [GATE 4] Kokoro Neural Speech Synthesis          : {'PASS [✓]' if g4_pass else 'FAIL [X]'} ({g4_status})")
dry_log(f"  [GATE 5] Downstream 40-Sample Benchmark          : {'PASS [✓]' if g5_pass else 'FAIL [X]'} (CER: {raw_cer_val:.2f}%, Wrong-Spoken: {lit_counts['wrong_spoken']}, No-Answer: {lit_counts['no_answer']})")
dry_log(f"  [GATE 6] Distillation Convergence (>=5% drop)   : {'PASS [✓]' if g6_pass else 'FAIL [X]'} (Val KD: {best_val_loss:.4f} vs Baseline: {step0_kd_val:.4f})")
dry_log("="*65)
dry_log(f">>> OVERALL FINANCIAL & TECHNICAL GATE DECISION: {'PROCEED TO TARGET PROCUREMENT' if overall_gate_pass else 'HALTED (CRITERIA NOT MET)'} <<<")
dry_log("="*65)

gate_report = {
    "git_commit_sha": GIT_COMMIT_SHA,
    "dry_run": DRY_RUN,
    "overall_gate_pass": overall_gate_pass,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "target_architecture": "Rockchip_RK3576",
    "som_module": "Firefly_CAM_3576Q38_LPDDR5",
    "gates": {
        "Gate_1_Optical_Resolution": {"passed": g1_pass, "px_per_char": round(px_per_char, 2), "type": "analytic"},
        "Gate_2_Spatial_Unshuffle": {
            "passed": g2_pass,
            "tokens": 144,
            "dim": 896,
            "measured_host_latency_ms": round(downsampler_latency_ms, 3),
            "host_latency_threshold_ms": 10.0
        },
        "Gate_3_Memory_Bandwidth": {
            "passed": g3_pass,
            "operating_bw_gb_s": round(operating_bw, 2),
            "safe_ceiling_gb_s": bus_ceiling,
            "weights_mb": round(weights_bytes / 1e6, 2),
            "scales_mb": round(scales_bytes / 1e6, 2),
            "kv_cache_mb": round(kv_cache_bytes / 1e6, 2),
            "total_traffic_mb_per_token": round(total_mb_per_tok, 2),
            "unit_system": "decimal_bytes_mb_gb",
            "type": "analytic"
        },
        "Gate_4_Kokoro_Voice": {
            "passed": g4_pass,
            "sample_rate_hz": None,
            "faster_than_realtime": None,
            "rtf": None,
            "status": g4_status
        },
        "Gate_5_Benchmark_Dataset": {
            "passed": g5_pass,
            "samples_evaluated": len(eval_subset),
            "literal_key_accuracy_pct": round(lit_acc, 2),
            "safety_accuracy_pct": round(saf_acc, 2),
            "unstripped_cer_pct": round(raw_cer_val, 2),
            "unstripped_wer_pct": round(raw_wer_val, 2),
            "wrong_spoken_entities_count": lit_counts["wrong_spoken"],
            "incomplete_count": lit_counts.get("incomplete", 0),
            "no_answer_count": lit_counts.get("no_answer", 0)
        },
        "Gate_6_Multimodal_Distillation": {
            "passed": g6_pass,
            "stage_1a_step0_caption_loss": round(step0_val_loss, 4),
            "stage_1a_final_caption_loss": round(final_stage1a_val, 4),
            "stage_1b_step0_kd_loss": round(step0_kd_val, 4),
            "stage_1b_best_kd_loss": round(best_val_loss, 4),
            "relative_loss_reduction_pct": round(((step0_kd_val - best_val_loss) / max(1e-5, step0_kd_val)) * 100.0, 2)
        }
    }
}

# Auditor Item A10: With DRY_RUN set, write to dry_run_gate_report.json
if DRY_RUN:
    report_filename = "dry_run_gate_report.json"
else:
    report_filename = "milestone_1_5_gate_report.json"

report_path = os.path.join(CHECKPOINT_DIR, report_filename)
with open(report_path, "w") as f:
    json.dump(gate_report, f, indent=2)

# Also write to repo root in DRY_RUN mode for easy auditing
if DRY_RUN:
    with open("dry_run_gate_report.json", "w") as f:
        json.dump(gate_report, f, indent=2)

dry_log(f"[✓] Gate Report Saved to: {report_path}")

PROJECT FRIDAY: MILESTONE 1.5 UNIFIED FINANCIAL & TECHNICAL GATE
  [GATE 1] Optical Resolution (>=10 px/char)      : PASS [✓] (12.8 px/char, Analytic)
  [GATE 2] Spatial Unshuffle (strictly 144 tokens) : PASS [✓]
  [GATE 3] Memory Bandwidth (<14.00 GB/s)          : PASS [✓] (5.75 GB/s @ 25 tok/s, Analytic)
  [GATE 4] Kokoro Neural Speech Synthesis          : FAIL [X] (NOT TESTED (Kokoro runtime not loaded in this session))
  [GATE 5] Downstream 40-Sample Benchmark          : FAIL [X] (CER: 6.24%, Wrong-Spoken: 0, No-Answer: 0)
  [GATE 6] Distillation Convergence (>=5% drop)   : PASS [✓] (Val KD: 1.5691 vs Baseline: 3.2788)
>>> OVERALL FINANCIAL & TECHNICAL GATE DECISION: HALTED (CRITERIA NOT MET) <<<
[✓] Gate Report Saved to: /kaggle/working/FRIDAY_weights/milestone_1_5_gate_report.json
